In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import math

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
content = path.read_bytes()

expected = json.loads(
    path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(content).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(content)
group_rows = []

for layer in plan["inventory"]:
    out_channels, in_channels, kh, kw = layer["weight_shape"]
    if math.prod(layer["weight_shape"]) != layer["weight_count"]:
        raise RuntimeError("Weight count mismatch.")

    if (kh, kw) == (3, 3):
        groups = out_channels * in_channels
        remainder = 0
    elif (kh, kw) == (1, 1):
        # Keep groups inside each output channel.
        groups = out_channels * (in_channels // 9)
        remainder = out_channels * (in_channels % 9)
    else:
        raise RuntimeError("Unsupported kernel shape.")

    grouped = groups * 9
    if grouped + remainder != layer["weight_count"]:
        raise RuntimeError("Grouping accounting mismatch.")

    group_rows.append({
        "layer": layer["name"],
        "groups": groups,
        "grouped_weights": grouped,
        "ungrouped_weights": remainder,
    })

total = sum(layer["weight_count"] for layer in plan["inventory"])
groups = sum(row["groups"] for row in group_rows)
grouped = sum(row["grouped_weights"] for row in group_rows)
remainder = sum(row["ungrouped_weights"] for row in group_rows)

print("Eligible layers:", len(group_rows))
print("Total logical INT8 weights:", total)
print("Nine-weight groups:", groups)
print("Grouped weights:", grouped)
print("Ungrouped tail weights:", remainder)
print("Grouped weight fraction:", grouped / total)

# Two three-bit masks per group: force-zero and force-one.
print("Ideal six-bit-mask storage, bytes:", (6 * groups + 7) // 8)
print("One-byte-per-group mask storage, bytes:", groups)
print("One-byte mask overhead / logical INT8 weights, %:", 100 * groups / total)
print("These estimates exclude headers and other metadata.")

print("\nLayers with ungrouped tails:")
for row in group_rows:
    if row["ungrouped_weights"]:
        print(row["layer"], "| tail weights:", row["ungrouped_weights"])

print("\nActual invariant-bit coverage has not been measured.")
print("No model loading, inference, or project file changes.")

Mounted at /content/drive
Eligible layers: 63
Total logical INT8 weights: 11146080
Nine-weight groups: 1235584
Grouped weights: 11120256
Ungrouped tail weights: 25824
Grouped weight fraction: 0.9976831316480772
Ideal six-bit-mask storage, bytes: 926688
One-byte-per-group mask storage, bytes: 1235584
One-byte mask overhead / logical INT8 weights, %: 11.08536812942308
These estimates exclude headers and other metadata.

Layers with ungrouped tails:
model.2.cv1.conv | tail weights: 64
model.2.cv2.conv | tail weights: 384
model.4.cv1.conv | tail weights: 256
model.4.cv2.conv | tail weights: 512
model.6.cv1.conv | tail weights: 1024
model.6.cv2.conv | tail weights: 2048
model.8.cv1.conv | tail weights: 4096
model.8.cv2.conv | tail weights: 1536
model.9.cv1.conv | tail weights: 2048
model.9.cv2.conv | tail weights: 3584
model.12.cv1.conv | tail weights: 768
model.12.cv2.conv | tail weights: 1536
model.15.cv1.conv | tail weights: 768
model.15.cv2.conv | tail weights: 384
model.18.cv1.conv | t

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()

expected = json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(plan_bytes).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(plan_bytes)
protocol = {
    "version": "1.0",
    "status": "FROZEN_BASELINE_BEFORE_INVARIANT_MASK_EXTRACTION",
    "reference_doi": "10.1007/s11227-024-06693-7",
    "implementation_claim": "documented_adaptation_not_exact_paper_reproduction",
    "representation": "05a_shared_logical_INT8_per_output_channel",
    "checkpoint_sha256": plan["checkpoint_sha256"],
    "clean_model_state_sha256": plan["clean_model_state_sha256"],
    "source_screening_plan_sha256": expected,
    "scope": "63_eligible_Conv2d_weight_tensors",
    "protected_candidate_bits": [4, 5, 6],
    "grouping": {
        "3x3": "one_9_weight_kernel_per_output_input_channel_pair",
        "1x1": "consecutive_groups_of_9_input_weights_within_each_output_channel",
        "order": "contiguous_OIHW_tensor_order",
        "tails": "leave_fewer_than_9_remaining_weights_unprotected",
        "cross_output_channel_grouping": False
    },
    "mask_definition": {
        "force_zero": "candidate_bit_is_zero_in_all_9_clean_weights",
        "force_one": "candidate_bit_is_one_in_all_9_clean_weights",
        "other_bits": "unprotected_and_unchanged",
        "zero_one_masks_must_be_disjoint": True,
        "storage": "one_uint8_per_group",
        "packing": (
            "bits_0_1_2_force_zero_for_weight_bits_4_5_6; "
            "bits_3_4_5_force_one_for_weight_bits_4_5_6; "
            "bits_6_7_reserved_zero"
        )
    },
    "operation": {
        "reference": "extract_once_from_verified_clean_integer_weights",
        "runtime": "scan_all_weights_in_the_selected_protected_groups",
        "correction": "set_only_protected_bits_to_the_stored_reference_value",
        "fault_location_provided_to_method": False,
        "clean_target_values_available_at_runtime": False
    },
    "trust_assumptions": [
        "reference_masks_and_group_metadata_are_trusted",
        "quantization_scales_are_trusted",
        "checker_and_correction_computation_are_trusted"
    ],
    "initial_checks": [
        "clean_weights_are_unchanged",
        "every_protected_single_bit_flip_is_exactly_restored",
        "unprotected_single_bit_flips_are_left_unchanged",
        "tails_are_unchanged",
        "mask_encoding_and_decoding_are_consistent",
        "signed_INT8_boundaries_are_handled_correctly"
    ],
    "coverage_reporting": {
        "grouped_weight_fraction": "not_fault_protection_coverage",
        "candidate_bit_coverage": "protected_weight_bit_positions_over_all_weight_positions_in_bits_4_5_6",
        "all_bit_coverage": "protected_weight_bit_positions_over_8_times_all_eligible_weights",
        "report_per_layer": True,
        "include_unprotected_tails_in_denominators": True
    },
    "cost_reporting": [
        "actual_mask_array_bytes",
        "actual_serialized_metadata_and_file_bytes",
        "ideal_6_bit_packed_size_as_theoretical_reference_only",
        "mask_construction_time",
        "scan_and_correction_time"
    ],
    "limitations": [
        "Not a new protection mechanism claim.",
        "No hardware implementation or native_INT8_inference claim.",
        "No AP improvement established by bit coverage alone.",
        "Trusted metadata faults and activation faults are outside scope.",
        "Protection allocation comparisons require a separate frozen protocol."
    ],
    "existing_05b_and_06b_protocols_changed": False
}

path = ROOT / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json"
content = (
    json.dumps(protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

path.parent.mkdir(parents=True, exist_ok=True)
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; nothing overwritten.")
    print("IDENTICAL PROTOCOL ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW PROTOCOL SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No mask extraction, model loading, or inference.")

NEW PROTOCOL SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook07a/invariant_bit_baseline_protocol_v1.json
SHA256: 74785069999784731bcc2917a5fd44fa8b698c102f76a9043b1f022b8303a799
No mask extraction, model loading, or inference.


In [ ]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

protected = ("torch", "numpy", "scikit-learn")
before = {name: version(name) for name in protected}

try:
    installed = version("ultralytics")
except PackageNotFoundError:
    installed = None

if installed is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps", "ultralytics==8.4.189",
        ],
        check=True,
    )
elif installed != "8.4.189":
    raise RuntimeError(
        f"Unexpected Ultralytics version: {installed}; nothing changed."
    )

after = {name: version(name) for name in protected}
if before != after:
    raise RuntimeError("Protected package versions changed.")

from ultralytics import YOLO
import torch

print("Ultralytics:", version("ultralytics"))
print("Protected versions unchanged:", after)
print("CUDA available:", torch.cuda.is_available())
print("YOLO import: PASS")
print("No checkpoint loading, mask extraction, or inference.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
Protected versions unchanged: {'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'scikit-learn': '1.6.1'}
CUDA available: False
YOLO import: PASS
No checkpoint loading, mask extraction, or inference.


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_extract_v1.py"

source = r'''"""Extract frozen invariant-bit masks on CPU; no COCO image inference."""
import hashlib
import importlib.util
import io
import json
import os
import platform
import time
from pathlib import Path

PROTOCOL_SHA = "74785069999784731bcc2917a5fd44fa8b698c102f76a9043b1f022b8303a799"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def publish(path, content):
    partial = path.with_name(path.name + ".part")
    if path.exists() or partial.exists():
        raise FileExistsError(f"Existing output requires review: {path.name}")
    with partial.open("xb") as stream:
        stream.write(content)
        stream.flush()
        os.fsync(stream.fileno())
    if partial.read_bytes() != content:
        raise RuntimeError("Write verification failed.")
    partial.rename(path)

def run(repo):
    import numpy as np
    import torch
    from importlib.metadata import version

    repo = Path(repo)
    out = repo / "results/notebook07a/invariant_masks_v1"
    if out.exists():
        raise FileExistsError("Output directory exists; inspect before rerunning.")

    protocol = json.loads(verified(
        repo / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json",
        PROTOCOL_SHA
    ))
    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    screen_path = repo / "src/screen05b_campaign_v1.py"
    verified(screen_path, plan["source_sha256"])
    spec = importlib.util.spec_from_file_location("screen07a_verified", screen_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != protocol["clean_model_state_sha256"]:
        raise RuntimeError("Shared clean model mismatch.")

    started = time.perf_counter()
    masks, inventory = {}, []
    popcount = np.array([int(i).bit_count() for i in range(256)], dtype=np.uint8)

    for index, item in enumerate(plan["inventory"]):
        name = item["name"]
        q = prepared.quantized[name][0].cpu().contiguous().numpy()
        if list(q.shape) != item["weight_shape"] or str(q.dtype) != "int8":
            raise RuntimeError("Quantized tensor shape or dtype mismatch.")
        o, i, kh, kw = q.shape
        unsigned = q.view(np.uint8)

        if (kh, kw) == (3, 3):
            groups = unsigned.reshape(-1, 9)
            tail = 0
        elif (kh, kw) == (1, 1):
            complete_inputs = (i // 9) * 9
            groups = unsigned.reshape(o, i)[:, :complete_inputs].reshape(-1, 9)
            tail = o * (i - complete_inputs)
        else:
            raise RuntimeError("Unsupported kernel shape.")

        all_one = np.bitwise_and.reduce(groups, axis=1) & 0x70
        all_zero = np.bitwise_not(
            np.bitwise_or.reduce(groups, axis=1)
        ) & 0x70
        if np.any(all_one & all_zero):
            raise RuntimeError("Conflicting masks.")

        packed = (
            (all_zero >> 4) | ((all_one >> 4) << 3)
        ).astype(np.uint8)
        if np.any(packed & 0xC0):
            raise RuntimeError("Reserved mask bits are not zero.")

        key = f"layer_{index:03d}"
        masks[key] = packed
        protected = int(popcount[packed].sum()) * 9
        count = int(q.size)
        if len(groups) * 9 + tail != count:
            raise RuntimeError("Grouping count mismatch.")

        inventory.append({
            "layer_id": index,
            "layer": name,
            "shape": list(q.shape),
            "array_key": key,
            "clean_integer_array_sha256": sha(q.tobytes()),
            "mask_array_sha256": sha(packed.tobytes()),
            "weights": count,
            "groups": len(groups),
            "tail_weights": tail,
            "mask_array_bytes": packed.nbytes,
            "protected_weight_bit_positions": protected,
            "candidate_bit_coverage": protected / (3 * count),
            "all_bit_coverage": protected / (8 * count),
            "protected_positions_by_bit": {
                str(bit): int(np.count_nonzero(
                    (all_zero | all_one) & (1 << bit)
                )) * 9
                for bit in (4, 5, 6)
            }
        })

    prepared._assert_state(prepared.clean_hash)
    extraction_seconds = time.perf_counter() - started
    if len(inventory) != 63:
        raise RuntimeError("Expected 63 layers.")

    buffer = io.BytesIO()
    np.savez_compressed(buffer, **masks)
    archive_bytes = buffer.getvalue()
    with np.load(io.BytesIO(archive_bytes), allow_pickle=False) as archive:
        if set(archive.files) != set(masks):
            raise RuntimeError("Mask archive keys mismatch.")
        for key, array in masks.items():
            if not np.array_equal(array, archive[key]):
                raise RuntimeError("Mask archive round-trip mismatch.")

    total = sum(row["weights"] for row in inventory)
    groups = sum(row["groups"] for row in inventory)
    protected = sum(row["protected_weight_bit_positions"] for row in inventory)
    payload = {
        "status": "MASKS_EXTRACTED_CORRECTION_TESTS_PENDING",
        "protocol_sha256": PROTOCOL_SHA,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "source_screen_sha256": plan["source_sha256"],
        "checkpoint_sha256": protocol["checkpoint_sha256"],
        "clean_model_state_sha256": prepared.clean_hash,
        "mask_archive_sha256": sha(archive_bytes),
        "environment": {
            "python": platform.python_version(),
            "packages": {p: version(p) for p in ("torch", "numpy", "ultralytics")},
            "device": "cpu",
            "torch_threads": torch.get_num_threads()
        },
        "totals": {
            "weights": total,
            "groups": groups,
            "tail_weights": sum(row["tail_weights"] for row in inventory),
            "mask_array_bytes": sum(array.nbytes for array in masks.values()),
            "ideal_six_bit_storage_bytes": (6 * groups + 7) // 8,
            "compressed_mask_archive_bytes": len(archive_bytes),
            "protected_weight_bit_positions": protected,
            "candidate_bit_coverage": protected / (3 * total),
            "all_bit_coverage": protected / (8 * total),
            "extraction_seconds_excluding_model_preparation": extraction_seconds
        },
        "inventory": inventory,
        "limitations": [
            "Trusted masks, metadata and scales.",
            "Actual protection coverage differs from grouped weight fraction.",
            "Compressed file size is not runtime mask memory.",
            "Correction correctness and AP effectiveness are not yet tested."
        ]
    }
    manifest_bytes = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()

    out.mkdir(parents=True, exist_ok=False)
    publish(out / "masks.npz", archive_bytes)
    publish(out / "manifest.json", manifest_bytes)

    print(json.dumps(payload["totals"], indent=2))
    print("Serialized manifest bytes:", len(manifest_bytes))
    print("Total archive + manifest bytes:", len(archive_bytes) + len(manifest_bytes))
    print("Saved:", out)
    print("Manifest SHA256:", sha(manifest_bytes))
    print("No COCO images, correction experiment, or 05b execution.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Extraction has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/invariant07a_extract_v1.py
Syntax: PASS
Extraction has not been executed.


In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_extract_v1.py"

spec = importlib.util.spec_from_file_location("invariant07a_extract_v1", path)
invariant07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(invariant07a)

mask_manifest07a = invariant07a.run(ROOT)

{
  "weights": 11146080,
  "groups": 1235584,
  "tail_weights": 25824,
  "mask_array_bytes": 1235584,
  "ideal_six_bit_storage_bytes": 926688,
  "compressed_mask_archive_bytes": 147693,
  "protected_weight_bit_positions": 1297125,
  "candidate_bit_coverage": 0.03879166487231386,
  "all_bit_coverage": 0.014546874327117695,
  "extraction_seconds_excluding_model_preparation": 0.28175375000000713
}
Serialized manifest bytes: 50418
Total archive + manifest bytes: 198111
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1
Manifest SHA256: bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a
No COCO images, correction experiment, or 05b execution.


In [ ]:
from pathlib import Path
import hashlib
import io
import json
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
FOLDER = ROOT / "results/notebook07a/invariant_masks_v1"

manifest_bytes = (FOLDER / "manifest.json").read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest() != (
    "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
):
    raise RuntimeError("Manifest checksum mismatch.")

envelope = json.loads(manifest_bytes)
manifest = envelope["payload"]
canonical = json.dumps(
    manifest, sort_keys=True, ensure_ascii=False,
    allow_nan=False, separators=(",", ":")
).encode("utf-8")
if hashlib.sha256(canonical).hexdigest() != envelope["payload_sha256"]:
    raise RuntimeError("Manifest payload mismatch.")

raw = (FOLDER / "masks.npz").read_bytes()
if hashlib.sha256(raw).hexdigest() != manifest["mask_archive_sha256"]:
    raise RuntimeError("Mask archive checksum mismatch.")

totals_by_bit = {bit: 0 for bit in (4, 5, 6)}
total_groups = 0
groups_with_protection = 0

with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
    expected_keys = {row["array_key"] for row in manifest["inventory"]}
    if set(archive.files) != expected_keys:
        raise RuntimeError("Unexpected mask arrays.")

    for row in manifest["inventory"]:
        packed = archive[row["array_key"]]
        if packed.dtype != np.uint8 or packed.shape != (row["groups"],):
            raise RuntimeError("Mask shape or dtype mismatch.")
        if hashlib.sha256(packed.tobytes()).hexdigest() != row["mask_array_sha256"]:
            raise RuntimeError("Layer mask checksum mismatch.")

        zero = packed & 0b111
        one = (packed >> 3) & 0b111

        if np.any(packed & 0b11000000) or np.any(zero & one):
            raise RuntimeError("Invalid or conflicting mask encoding.")
        if not np.array_equal(packed, zero | (one << 3)):
            raise RuntimeError("Mask round-trip mismatch.")

        protected = zero | one
        total_groups += len(packed)
        groups_with_protection += int(np.count_nonzero(protected))

        for offset, bit in enumerate((4, 5, 6)):
            positions = int(np.count_nonzero(protected & (1 << offset))) * 9
            if positions != row["protected_positions_by_bit"][str(bit)]:
                raise RuntimeError("Per-bit coverage mismatch.")
            totals_by_bit[bit] += positions

if sum(totals_by_bit.values()) != manifest["totals"]["protected_weight_bit_positions"]:
    raise RuntimeError("Total coverage mismatch.")

weights = manifest["totals"]["weights"]
print("Mask integrity and encoding: PASS")
print("Groups with at least one protected bit:", groups_with_protection)
print("Total groups:", total_groups)

for bit, count in totals_by_bit.items():
    print(
        f"Bit {bit}: protected weight positions={count:,} | "
        f"coverage within this bit={100 * count / weights:.6f}%"
    )

print("No weight correction or COCO inference executed.")

Mask integrity and encoding: PASS
Groups with at least one protected bit: 100382
Total groups: 1235584
Bit 4: protected weight positions=117,081 | coverage within this bit=1.050423%
Bit 5: protected weight positions=334,899 | coverage within this bit=3.004635%
Bit 6: protected weight positions=845,145 | coverage within this bit=7.582442%
No weight correction or COCO inference executed.


In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_check_v1.py"

source = r'''"""Check invariant-mask correction on CPU; no COCO image inference."""
import hashlib
import importlib.util
import io
import json
import time
from pathlib import Path

MANIFEST_SHA = "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def grouped(values):
    o, i, kh, kw = values.shape
    if (kh, kw) == (3, 3):
        return values.reshape(-1, 9)
    if (kh, kw) == (1, 1):
        end = (i // 9) * 9
        return values.reshape(o, i)[:, :end].reshape(-1, 9)
    raise ValueError("Unsupported kernel shape.")

def correct(observed, packed):
    """Use observed weights and trusted masks only; no clean-weight input."""
    import numpy as np
    if observed.dtype != np.int8 or packed.dtype != np.uint8:
        raise ValueError("Expected int8 weights and uint8 masks.")
    result = observed.copy()
    unsigned = result.view(np.uint8)
    blocks = grouped(unsigned)
    if packed.shape != (len(blocks),):
        raise ValueError("Mask count mismatch.")
    zero = ((packed & 7) << 4).astype(np.uint8)
    one = (((packed >> 3) & 7) << 4).astype(np.uint8)
    if np.any(zero & one) or np.any(packed & 0xC0):
        raise ValueError("Invalid masks.")
    fixed = (blocks & np.bitwise_not(zero[:, None])) | one[:, None]
    o, i, kh, kw = result.shape
    if (kh, kw) == (3, 3):
        unsigned.reshape(-1, 9)[:] = fixed
    else:
        end = (i // 9) * 9
        unsigned.reshape(o, i)[:, :end] = fixed.reshape(o, end)
    return result

def run(repo):
    import numpy as np
    import torch
    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    destination = folder / "correctness_v1.json"
    if destination.exists():
        raise FileExistsError("Correctness report exists; inspect before rerunning.")

    envelope = json.loads(verified(folder / "manifest.json", MANIFEST_SHA))
    manifest = envelope["payload"]
    if sha(canonical(manifest)) != envelope["payload_sha256"]:
        raise RuntimeError("Manifest payload mismatch.")
    protocol = json.loads(verified(
        repo / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json",
        manifest["protocol_sha256"]
    ))
    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    screen_path = repo / "src/screen05b_campaign_v1.py"
    verified(screen_path, manifest["source_screen_sha256"])
    if plan["source_sha256"] != manifest["source_screen_sha256"]:
        raise RuntimeError("Source provenance mismatch.")
    spec = importlib.util.spec_from_file_location("screen07a_check", screen_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != manifest["clean_model_state_sha256"]:
        raise RuntimeError("Clean model mismatch.")

    raw = verified(folder / "masks.npz", manifest["mask_archive_sha256"])
    cases = recovered = unchanged_unprotected = clean_weights = 0
    rows = []
    started = time.perf_counter()

    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            q = prepared.quantized[item["layer"]][0].cpu().contiguous().numpy()
            packed = archive[item["array_key"]].copy()
            if sha(q.tobytes()) != item["clean_integer_array_sha256"]:
                raise RuntimeError("Clean integer tensor mismatch.")
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Layer mask checksum mismatch.")

            # Independently reconstruct invariant flags from all nine weights.
            blocks = grouped(q.view(np.uint8))
            expected = np.zeros(len(blocks), dtype=np.uint8)
            for offset, bit in enumerate((4, 5, 6)):
                values = (blocks >> bit) & 1
                expected |= np.all(values == 0, axis=1).astype(np.uint8) << offset
                expected |= np.all(values == 1, axis=1).astype(np.uint8) << (offset + 3)
            if not np.array_equal(packed, expected):
                raise RuntimeError("Masks do not match clean group invariants.")

            if not np.array_equal(correct(q, packed), q):
                raise RuntimeError("Clean weights were changed.")
            clean_weights += q.size
            layer_recovered = 0

            for bit in range(8):
                # Batch independent position-wise flips under fixed masks.
                faulty = (q.view(np.uint8) ^ np.uint8(1 << bit)).view(np.int8)
                final = correct(faulty, packed)
                protected_positions = np.zeros(q.shape, dtype=bool)

                if bit in (4, 5, 6):
                    offset = bit - 4
                    flags = (((packed & 7) | ((packed >> 3) & 7)) >> offset) & 1
                    expanded = np.repeat(flags.astype(bool), 9)
                    o, i, kh, kw = q.shape
                    if (kh, kw) == (3, 3):
                        protected_positions.reshape(-1)[:] = expanded
                    else:
                        end = (i // 9) * 9
                        protected_positions.reshape(o, i)[:, :end] = expanded.reshape(o, end)

                if not np.array_equal(final[protected_positions], q[protected_positions]):
                    raise RuntimeError("Protected flip was not restored.")
                if not np.array_equal(final[~protected_positions], faulty[~protected_positions]):
                    raise RuntimeError("Unprotected flip or tail was altered.")

                count = int(protected_positions.sum())
                cases += q.size
                recovered += count
                layer_recovered += count
                unchanged_unprotected += q.size - count

            if layer_recovered != item["protected_weight_bit_positions"]:
                raise RuntimeError("Layer recovery count mismatch.")
            rows.append({
                "layer": item["layer"],
                "position_bit_cases": int(q.size * 8),
                "exactly_recovered_protected_cases": layer_recovered
            })

    prepared._assert_state(prepared.clean_hash)
    if recovered != manifest["totals"]["protected_weight_bit_positions"]:
        raise RuntimeError("Total recovery count mismatch.")
    payload = {
        "status": "CORRECTION_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION",
        "manifest_sha256": MANIFEST_SHA,
        "protocol_sha256": manifest["protocol_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "clean_weights_checked_unchanged": int(clean_weights),
        "position_bit_cases_checked": int(cases),
        "protected_cases_exactly_recovered": int(recovered),
        "unprotected_cases_left_unchanged": int(unchanged_unprotected),
        "check_seconds_excluding_model_preparation": time.perf_counter() - started,
        "method": "Vectorized position-wise checks; fixed masks make corrections separable.",
        "limits": [
            "Not COCO inference or an AP benefit measurement.",
            "Trusted masks, scales, metadata and checker.",
            "This does not validate hardware fault handling or scan performance."
        ],
        "layers": rows
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    for key in (
        "status", "clean_weights_checked_unchanged",
        "position_bit_cases_checked", "protected_cases_exactly_recovered",
        "unprotected_cases_left_unchanged",
        "check_seconds_excluding_model_preparation"
    ):
        print(f"{key}: {payload[key]}")
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("invariant07a_check_v1", path)
check07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(check07a)
correctness07a = check07a.run(ROOT)

status: CORRECTION_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION
clean_weights_checked_unchanged: 11146080
position_bit_cases_checked: 89168640
protected_cases_exactly_recovered: 1297125
unprotected_cases_left_unchanged: 87871515
check_seconds_excluding_model_preparation: 1.1109702830000288
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/correctness_v1.json
SHA256: b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206


In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_compare_v1.py"

source = r'''"""Retrospective comparison on existing 06b test targets; no inference."""
import hashlib
import io
import json
from pathlib import Path

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def record(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record checksum mismatch.")
    return envelope["payload"]

def run(repo):
    import numpy as np
    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    manifest = record(verified(
        folder / "manifest.json",
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
    ))
    correctness = record(verified(
        folder / "correctness_v1.json",
        "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206"
    ))
    if correctness["manifest_sha256"] != (
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
    ):
        raise RuntimeError("Correctness provenance mismatch.")

    source06a = repo / "results/notebook06a"
    session = record(
        (source06a / "masked_bit_comparison_v1/session.json").read_bytes()
    )
    dataset_bytes = verified(
        source06a / "masked_bit_dataset_v1/dataset.npz",
        session["dataset_sha256"]
    )
    with np.load(io.BytesIO(dataset_bytes), allow_pickle=False) as archive:
        test = {key: archive[f"test_{key}"].copy()
                for key in ("bits", "layer", "channel", "flat_index")}

    folder06b = repo / "results/notebook06b/evaluation_v1"
    summary = record((folder06b / "summary.json").read_bytes())
    if sha(canonical(session)) != summary["source_session_fingerprint"]:
        raise RuntimeError("06a/06b session mismatch.")
    outcome_bytes = verified(
        folder06b / "outcomes.npz", summary["outcomes_sha256"]
    )
    with np.load(io.BytesIO(outcome_bytes), allow_pickle=False) as archive:
        outcomes = {key: archive[key].copy() for key in archive.files}
    for key in ("layer", "channel", "flat_index"):
        if not np.array_equal(test[key], outcomes[key]):
            raise RuntimeError("Test target alignment mismatch.")

    n = len(test["bits"])
    recovered = np.zeros((n, 8), dtype=bool)
    mask_bytes = verified(folder / "masks.npz", manifest["mask_archive_sha256"])

    with np.load(io.BytesIO(mask_bytes), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            selected = np.flatnonzero(test["layer"] == item["layer_id"])
            o, i, kh, kw = item["shape"]
            flat = test["flat_index"][selected]
            if np.any(flat < 0) or np.any(flat >= o * i * kh * kw):
                raise RuntimeError("Invalid target index.")
            channel = flat // (i * kh * kw)
            if not np.array_equal(channel, test["channel"][selected]):
                raise RuntimeError("Target channel mismatch.")

            if (kh, kw) == (3, 3):
                valid = np.ones(len(selected), dtype=bool)
                group_index = flat // 9
            elif (kh, kw) == (1, 1):
                within = flat % i
                valid = within < (i // 9) * 9
                group_index = channel * (i // 9) + within // 9
            else:
                raise RuntimeError("Unsupported kernel.")

            packed = archive[item["array_key"]]
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Layer mask mismatch.")
            rows = selected[valid]
            codes = packed[group_index[valid]]
            zero, one = codes & 7, (codes >> 3) & 7

            for offset, bit in enumerate((4, 5, 6)):
                force_zero = ((zero >> offset) & 1).astype(bool)
                force_one = ((one >> offset) & 1).astype(bool)
                clean_bits = test["bits"][rows, bit]
                if np.any(clean_bits[force_zero] != 0) or np.any(clean_bits[force_one] != 1):
                    raise RuntimeError("Mask disagrees with clean sampled targets.")
                recovered[rows, bit] = force_zero | force_one

    methods = {
        "invariant_group_baseline": {
            "exact_recovery_fraction": float(recovered.mean()),
            "exact_recovery_by_bit": recovered.mean(axis=0).tolist(),
            "clean_false_intervention_fraction": 0.0,
            "clean_control_basis": "Full clean-weight functional check and invariant agreement.",
        }
    }
    for method in ("C", "D"):
        values = outcomes[f"{method}_exact_recovery"]
        methods[method] = {
            "exact_recovery_fraction": float(values.mean()),
            "exact_recovery_by_bit": values.mean(axis=0).tolist(),
            "clean_false_intervention_fraction": float(
                outcomes[f"{method}_clean_intervention"].mean()
            )
        }

    payload = {
        "status": "RETROSPECTIVE_COMMON_TARGET_COMPARISON_NOT_EQUAL_BUDGET",
        "targets": n,
        "single_bit_cases_per_method": n * 8,
        "dataset_sha256": session["dataset_sha256"],
        "outcomes_sha256": summary["outcomes_sha256"],
        "mask_archive_sha256": manifest["mask_archive_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "methods": methods,
        "limitations": [
            "Existing inspected 06b test targets; exploratory comparison.",
            "Recovery of the invariant baseline derived from verified bit coverage.",
            "Predictor and mask memory budgets are not matched.",
            "Uniformity over sampled targets and bits is not whole-network uniform sampling.",
            "No new COCO inference or AP benefit established."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    destination = folder / "comparison_on_06b_targets_v1.json"
    if destination.exists():
        if destination.read_bytes() != content:
            raise RuntimeError("Existing comparison differs; nothing overwritten.")
    else:
        with destination.open("xb") as stream:
            stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    print("Common test targets:", n)
    print("Single-bit cases per method:", n * 8)
    for method, result in methods.items():
        print(
            f"{method}: exact recovery={100 * result['exact_recovery_fraction']:.6f}% | "
            f"clean false intervention={100 * result['clean_false_intervention_fraction']:.6f}%"
        )
        print("Recovery by bit, %:",
              [round(100 * x, 6) for x in result["exact_recovery_by_bit"]])
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("invariant07a_compare_v1", path)
compare07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(compare07a)
comparison07a = compare07a.run(ROOT)

Common test targets: 16061
Single-bit cases per method: 128488
invariant_group_baseline: exact recovery=1.180655% | clean false intervention=0.000000%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.0, 1.351099, 2.776913, 5.317228, 0.0]
C: exact recovery=17.424195% | clean false intervention=0.958844%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.211693, 4.495361, 15.490941, 48.602204, 70.593363]
D: exact recovery=12.257954% | clean false intervention=0.765830%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.0, 0.0, 1.351099, 23.852811, 72.859722]
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/comparison_on_06b_targets_v1.json
SHA256: 3410026a67e06483f4fff3e6823c9a7dcf47ee6c9c6d658693937983214e9254


In [ ]:
from pathlib import Path
import hashlib
import io
import json
import joblib

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
FOLDER = ROOT / "results/notebook06a/masked_bit_comparison_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    envelope = json.loads(path.read_bytes())
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {path.name}")
    return payload

session = read_record(FOLDER / "session.json")
fingerprint = hashlib.sha256(canonical(session)).hexdigest()
sizes = {
    method: {"compressed_bytes": 0, "uncompressed_bytes": 0}
    for method in ("C", "D")
}

for bit in range(8):
    record = read_record(FOLDER / f"bit_{bit}_result.json")
    if record["bit"] != bit or record["session_fingerprint"] != fingerprint:
        raise RuntimeError("Model session mismatch.")

    content = (FOLDER / f"bit_{bit}_models.joblib").read_bytes()
    if hashlib.sha256(content).hexdigest() != record["models_sha256"]:
        raise RuntimeError("Model checksum mismatch.")

    # Load only verified, locally produced artifacts.
    models = joblib.load(io.BytesIO(content))
    for method in ("C", "D"):
        for compression, key in (
            (3, "compressed_bytes"),
            (0, "uncompressed_bytes"),
        ):
            buffer = io.BytesIO()
            joblib.dump(models[method], buffer, compress=compression)
            sizes[method][key] += buffer.tell()

print("Eight predictors per method:")
print(json.dumps(sizes, indent=2))
print("\nSerialization sizes are not live RAM measurements.")
print("Thresholds, feature construction, metadata and scan costs are excluded.")
print("No predictions, fitting, or file writes.")

Eight predictors per method:
{
  "C": {
    "compressed_bytes": 582653,
    "uncompressed_bytes": 1618336
  },
  "D": {
    "compressed_bytes": 490424,
    "uncompressed_bytes": 1521824
  }
}

Serialization sizes are not live RAM measurements.
Thresholds, feature construction, metadata and scan costs are excluded.
No predictions, fitting, or file writes.


In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = ROOT / "results/notebook07a/invariant_masks_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def verified_record(name, expected):
    content = (SOURCE / name).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError(f"File checksum mismatch: {name}")
    envelope = json.loads(content)
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {name}")
    return payload

hashes = {
    "manifest.json":
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a",
    "correctness_v1.json":
        "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206",
    "comparison_on_06b_targets_v1.json":
        "3410026a67e06483f4fff3e6823c9a7dcf47ee6c9c6d658693937983214e9254",
}
records = {
    name: verified_record(name, expected)
    for name, expected in hashes.items()
}

if set(sizes) != {"C", "D"}:
    raise RuntimeError("Serialization measurements are unavailable.")
for method in ("C", "D"):
    for key in ("compressed_bytes", "uncompressed_bytes"):
        if type(sizes[method][key]) is not int or sizes[method][key] <= 0:
            raise RuntimeError("Invalid serialization measurement.")

manifest = records["manifest.json"]
comparison = records["comparison_on_06b_targets_v1.json"]

report = {
    "status": "07A_FUNCTIONAL_BASELINE_COMPLETE_COST_AND_AP_STUDIES_PENDING",
    "source_file_sha256": hashes,
    "protocol_sha256": manifest["protocol_sha256"],
    "invariant_baseline_totals": manifest["totals"],
    "functional_checks": {
        key: records["correctness_v1.json"][key]
        for key in (
            "status",
            "clean_weights_checked_unchanged",
            "position_bit_cases_checked",
            "protected_cases_exactly_recovered",
            "unprotected_cases_left_unchanged",
        )
    },
    "common_test_targets": comparison["targets"],
    "common_target_methods": comparison["methods"],
    "predictor_serialization_bytes": sizes,
    "interpretation": [
        "C has higher exact recovery on the common inspected targets.",
        "The invariant baseline leaves clean weights unchanged under trusted masks.",
        "The baseline targets bits 4-6; C and D inspect all eight bits.",
        "Masks use clean reference weights, including test-channel weights.",
        "Predictor training uses the separate training-channel partition.",
        "Storage budgets and information assumptions differ.",
        "Compressed storage, uncompressed serialization and live RAM are distinct.",
        "No overall winner, independent confirmation or AP benefit is established."
    ],
    "pending": [
        "Full scan and correction timing.",
        "Live memory and feature-construction costs.",
        "A separate frozen protocol for equal-budget comparison.",
        "End-to-end evaluation of the invariant baseline.",
        "Git review and publication of new session files."
    ]
}

envelope = {
    "payload_sha256": hashlib.sha256(canonical(report)).hexdigest(),
    "payload": report,
}
content = (
    json.dumps(envelope, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination = ROOT / "docs/research_reports/notebook07a_baseline_summary_v1.json"
destination.parent.mkdir(parents=True, exist_ok=True)

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing report differs; nothing overwritten.")
    print("IDENTICAL REPORT ALREADY EXISTS")
else:
    with destination.open("xb") as stream:
        stream.write(content)
    print("NEW REPORT SAVED")

assert destination.read_bytes() == content
print("Path:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No experiments, model changes, commit, or push.")

NEW REPORT SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook07a_baseline_summary_v1.json
SHA256: 7cb1b0a831ed44ebbac3e3f91a7e6d64b963f427ab9fa479e96608be7348d10d
No experiments, model changes, commit, or push.


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_timing_v1.py"

source = r'''"""CPU timing of complete NumPy mask scans; not inference latency."""
import hashlib
import importlib.util
import io
import json
import platform
import time
from pathlib import Path

MANIFEST_SHA = "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
CHECK_SHA = "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206"
WARMUPS = 3
REPEATS = 20

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Payload checksum mismatch.")
    return envelope["payload"]

def load_module(name, path, expected):
    verified(path, expected)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def run(repo):
    import numpy as np
    import torch
    from importlib.metadata import version
    from threadpoolctl import threadpool_limits

    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    destination = folder / "cpu_scan_timing_v1.json"
    if destination.exists():
        raise FileExistsError("Timing report exists; inspect before rerunning.")
    manifest = unpack(verified(folder / "manifest.json", MANIFEST_SHA))
    checks = unpack(verified(folder / "correctness_v1.json", CHECK_SHA))
    checker = load_module(
        "verified07a_corrector", repo / "src/invariant07a_check_v1.py",
        checks["module_sha256"]
    )
    screen = load_module(
        "verified07a_timing_screen", repo / "src/screen05b_campaign_v1.py",
        manifest["source_screen_sha256"]
    )
    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != manifest["clean_model_state_sha256"]:
        raise RuntimeError("Clean state mismatch.")

    raw = verified(folder / "masks.npz", manifest["mask_archive_sha256"])
    clean, masks = [], []
    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            q = prepared.quantized[item["layer"]][0].cpu().contiguous().numpy().copy()
            packed = archive[item["array_key"]].copy()
            if sha(q.tobytes()) != item["clean_integer_array_sha256"]:
                raise RuntimeError("Clean tensor mismatch.")
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Mask mismatch.")
            clean.append(q)
            masks.append(packed)

    faulty = list(clean)
    selected_fault = None
    for layer_index, (q, packed) in enumerate(zip(clean, masks)):
        protected = (packed & 7) | ((packed >> 3) & 7)
        available = np.flatnonzero(protected)
        if not len(available):
            continue
        group = int(available[0])
        offset = next(b for b in range(3) if int(protected[group]) & (1 << b))
        bit = offset + 4
        o, i, kh, kw = q.shape
        if (kh, kw) == (3, 3):
            flat = group * 9
        else:
            per_channel = i // 9
            flat = (group // per_channel) * i + (group % per_channel) * 9
        faulty[layer_index] = q.copy()
        faulty[layer_index].view(np.uint8).reshape(-1)[flat] ^= np.uint8(1 << bit)
        selected_fault = {
            "layer": manifest["inventory"][layer_index]["layer"],
            "flat_index": flat, "bit": bit,
            "selection": "first_protected_position_to_exercise_correction_only"
        }
        break
    if selected_fault is None:
        raise RuntimeError("No protected fault available.")

    def scan(arrays):
        return [checker.correct(q, mask) for q, mask in zip(arrays, masks)]

    results = {}
    with threadpool_limits(limits=1):
        for label, arrays in (("clean_scan", clean), ("one_protected_fault_scan", faulty)):
            for _ in range(WARMUPS):
                output = scan(arrays)
                del output
            times = []
            for _ in range(REPEATS):
                started = time.perf_counter()
                output = scan(arrays)
                times.append(time.perf_counter() - started)
                # Validation and output deletion are outside the timed interval.
                if not all(np.array_equal(a, b) for a, b in zip(output, clean)):
                    raise RuntimeError("Scan output differs from clean reference.")
                del output
            results[label] = {
                "seconds": times,
                "median_seconds": float(np.median(times)),
                "min_seconds": float(np.min(times)),
                "empirical_p95_seconds": float(np.quantile(times, 0.95))
            }

    prepared._assert_state(prepared.clean_hash)
    payload = {
        "status": "CPU_NUMPY_SCAN_TIMING_COMPLETE_NOT_INFERENCE_OVERHEAD",
        "manifest_sha256": MANIFEST_SHA,
        "correctness_file_sha256": CHECK_SHA,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "warmups": WARMUPS, "repeats": REPEATS,
        "order": ["clean_scan", "one_protected_fault_scan"],
        "weights_scanned_per_repeat": sum(q.size for q in clean),
        "input_array_bytes": sum(q.nbytes for q in clean),
        "mask_array_bytes": sum(m.nbytes for m in masks),
        "output_array_bytes_per_scan": sum(q.nbytes for q in clean),
        "selected_fault": selected_fault,
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "numpy": version("numpy"), "torch": version("torch"),
            "threadpool_limit": 1
        },
        "results": results,
        "limitations": [
            "In-memory CPU NumPy implementation; outputs allocate new arrays.",
            "Model preparation, file reads, validation and output deletion excluded.",
            "Temporary allocation peaks and live RAM not measured.",
            "Warm-cache repeated measurements; not deployment latency guarantees.",
            "Selected protected fault is a functional timing workload, not an efficacy sample.",
            "No measured impact on complete YOLO inference latency."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    for label, result in results.items():
        print(
            f"{label}: median={result['median_seconds']:.6f}s | "
            f"empirical p95={result['empirical_p95_seconds']:.6f}s"
        )
    print("Weights per scan:", payload["weights_scanned_per_repeat"])
    print("Selected timing fault:", selected_fault)
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Timing has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/invariant07a_timing_v1.py
Syntax: PASS
Timing has not been executed.


In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_timing_v1.py"

spec = importlib.util.spec_from_file_location("invariant07a_timing_v1", path)
timing07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(timing07a)

timing_result07a = timing07a.run(ROOT)

clean_scan: median=0.044503s | empirical p95=0.055701s
one_protected_fault_scan: median=0.046084s | empirical p95=0.050897s
Weights per scan: 11146080
Selected timing fault: {'layer': 'model.0.conv', 'flat_index': 36, 'bit': 6, 'selection': 'first_protected_position_to_exercise_correction_only'}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json
SHA256: 6972ab2c57f074358dd506a6cd79e8e9497658ed29fd7fbb88d60a55455deda9


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

commands = [
    ["git", "rev-parse", "HEAD"],
    ["git", "--no-optional-locks", "status", "--short",
     "--untracked-files=normal"],
    ["git", "diff", "--stat"],
]

for command in commands:
    print("\nCOMMAND:", " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=ROOT,
            capture_output=True,
            text=True,
            timeout=120,
        )
        print("Return code:", result.returncode)
        print(result.stdout.strip() or "(no output)")
        if result.stderr.strip():
            print("STDERR:", result.stderr.strip())
        if result.returncode != 0:
            break
    except subprocess.TimeoutExpired:
        print("TIMEOUT: command exceeded 120 seconds; nothing changed.")
        break

print("\nRead-only inspection; no staging, commit, or push.")


COMMAND: git rev-parse HEAD
Return code: 0
97aca930e3c9fdb47e714e9d49decf4778d894c0

COMMAND: git --no-optional-locks status --short --untracked-files=normal
Return code: 0
?? docs/research_reports/notebook05b_screening_v1/progress_01600.json
?? docs/research_reports/notebook05b_screening_v1/progress_01604.json
?? docs/research_reports/notebook05b_screening_v1/progress_01650.json
?? docs/research_reports/notebook05b_screening_v1/progress_01700.json
?? docs/research_reports/notebook05b_screening_v1/progress_01704.json
?? docs/research_reports/notebook06b_end_to_end_v1/
?? docs/research_reports/notebook06b_per_layer_v1.csv
?? docs/research_reports/notebook06b_pilot_v1.md
?? docs/research_reports/notebook07a_baseline_summary_v1.json
?? experiments/notebook06b/
?? experiments/notebook07a/
?? src/calibration06b_v1.py
?? src/decisions06b_e2e_v1.py
?? src/end_to_end06b_v1.py
?? src/evaluation06b_v1.py
?? src/invariant07a_check_v1.py
?? src/invariant07a_compare_v1.py
?? src/invariant07a_extra

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

print("Ignore rules for representative result files:")
paths = [
    "results/notebook06b/calibration_v1.json",
    "results/notebook06b/evaluation_v1/summary.json",
    "results/notebook06b/end_to_end_pilot_v1/session.json.gz",
    "results/notebook07a/invariant_masks_v1/manifest.json",
    "results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json",
]
result = subprocess.run(
    ["git", "check-ignore", "-v", "--", *paths],
    cwd=ROOT, capture_output=True, text=True, timeout=30,
)
print(result.stdout.strip() or "(no matching ignore rules)")
if result.stderr.strip():
    print("STDERR:", result.stderr.strip())
print("Return code:", result.returncode)

print("\nSaved notebook candidates:")
directories = [
    ROOT / "notebooks",
    Path("/content/drive/MyDrive/Colab Notebooks"),
]
for directory in directories:
    print("\nDirectory:", directory)
    if not directory.is_dir():
        print("NOT FOUND")
        continue
    matches = [
        path for path in directory.glob("*.ipynb")
        if path.name.startswith(("06b", "07a"))
    ]
    for path in sorted(matches):
        print(path.name, "| bytes:", path.stat().st_size)
    if not matches:
        print("NO MATCHES")

print("\nNew protocol/report files:")
for relative in (
    "experiments/notebook06b",
    "experiments/notebook07a",
    "docs/research_reports/notebook06b_end_to_end_v1",
):
    directory = ROOT / relative
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            print(path.relative_to(ROOT), "| bytes:", path.stat().st_size)

print("\nRead-only; no copying, staging, commit, or push.")

Ignore rules for representative result files:
.gitignore:24:results/*	results/notebook06b/calibration_v1.json
.gitignore:24:results/*	results/notebook06b/evaluation_v1/summary.json
.gitignore:24:results/*	results/notebook06b/end_to_end_pilot_v1/session.json.gz
.gitignore:24:results/*	results/notebook07a/invariant_masks_v1/manifest.json
.gitignore:24:results/*	results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json
Return code: 0

Saved notebook candidates:

Directory: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/notebooks
NO MATCHES

Directory: /content/drive/MyDrive/Colab Notebooks
06b_detection_repair_pilot.ipynb | bytes: 284647
07a_invariant_bit_protectionfrom google.colab import drive from pathlib import Path import hashlib import json import math  ~.mount(" content drive")  ROOT = Path(" content drive MyDrive YOLOv8_Fault_Tolerance code") path = ROOT   "experiments noteb.ipynb | bytes: 72128

New protocol/report files:
experiments/notebook06b/detection_repair_pilot_pr

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = Path("/content/drive/MyDrive/Colab Notebooks")
DEST = ROOT / "notebooks"

source06b = SOURCE / "06b_detection_repair_pilot.ipynb"
candidates07a = list(SOURCE.glob("07a*.ipynb"))

if not source06b.is_file():
    raise FileNotFoundError("06b source notebook not found.")
if len(candidates07a) != 1:
    raise RuntimeError(
        f"Expected one 07a notebook; found {len(candidates07a)}. Nothing copied."
    )

pairs = [
    (source06b, DEST / "06b_detection_repair_pilot.ipynb"),
    (candidates07a[0], DEST / "07a_invariant_bit_protection.ipynb"),
]

prepared = []
for source, destination in pairs:
    content = source.read_bytes()
    notebook = json.loads(content)
    if notebook.get("nbformat") != 4 or not isinstance(notebook.get("cells"), list):
        raise RuntimeError(f"Invalid notebook: {source.name}")
    if destination.exists() and destination.read_bytes() != content:
        raise RuntimeError(
            f"Different destination already exists: {destination.name}. Nothing copied."
        )
    prepared.append((destination, content, notebook))

DEST.mkdir(parents=True, exist_ok=True)
for destination, content, notebook in prepared:
    if destination.exists():
        print("IDENTICAL COPY ALREADY EXISTS:", destination.name)
    else:
        with destination.open("xb") as stream:
            stream.write(content)
        print("COPIED:", destination.name)

    if destination.read_bytes() != content:
        raise RuntimeError("Copy verification failed.")

    cells = notebook["cells"]
    error_outputs = sum(
        output.get("output_type") == "error"
        for cell in cells
        for output in cell.get("outputs", [])
    )
    print("Cells:", len(cells))
    print("Saved error outputs:", error_outputs)
    print("SHA256:", hashlib.sha256(content).hexdigest())

print("\nOriginal notebooks preserved; no staging, commit, or push.")

COPIED: 06b_detection_repair_pilot.ipynb
Cells: 35
Saved error outputs: 0
SHA256: 2ba24b7f8469570db8aca270ef477521ef353be0ce210afdcef08f6b02b86d27
COPIED: 07a_invariant_bit_protection.ipynb
Cells: 14
Saved error outputs: 0
SHA256: cbd7a6f5dddf77bf1241ccb5a279d3db88da9dadeefcc01b0a174fab1b9a0a9a

Original notebooks preserved; no staging, commit, or push.


In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

items = [
    ("results/notebook06b/calibration_v1.json",
     "notebook06b_evidence_v1/calibration_v1.json",
     "87bfe4072cc4f8b1eb90fe961d4c348552f674223d6d49a85f5e7e0c7e1931e0"),
    ("results/notebook06b/evaluation_v1/summary.json",
     "notebook06b_evidence_v1/weight_evaluation_v1.json", None),
    ("results/notebook06b/evaluation_v1/paired_uncertainty_v1.json",
     "notebook06b_evidence_v1/paired_uncertainty_v1.json", None),
    ("results/notebook06b/end_to_end_decisions_v1.json",
     "notebook06b_evidence_v1/end_to_end_decisions_v1.json",
     "89a9a3c3e2b4f8187089325a765b1a715b7059d40ed8446fe1f08e3881fab2c0"),
    ("results/notebook07a/invariant_masks_v1/manifest.json",
     "notebook07a_evidence_v1/mask_manifest_v1.json",
     "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"),
    ("results/notebook07a/invariant_masks_v1/correctness_v1.json",
     "notebook07a_evidence_v1/correctness_v1.json",
     "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206"),
    ("results/notebook07a/invariant_masks_v1/comparison_on_06b_targets_v1.json",
     "notebook07a_evidence_v1/comparison_on_06b_targets_v1.json",
     "3410026a67e06483f4fff3e6823c9a7dcf47ee6c9c6d658693937983214e9254"),
    ("results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json",
     "notebook07a_evidence_v1/cpu_scan_timing_v1.json",
     "6972ab2c57f074358dd506a6cd79e8e9497658ed29fd7fbb88d60a55455deda9"),
]

prepared = []
for relative, destination_relative, expected in items:
    source = ROOT / relative
    destination = ROOT / "docs/research_reports" / destination_relative
    content = source.read_bytes()

    if expected is not None and sha(content) != expected:
        raise RuntimeError(f"Source checksum mismatch: {relative}")

    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {relative}")

    if destination.exists() and destination.read_bytes() != content:
        raise RuntimeError(f"Different destination exists: {destination_relative}")

    prepared.append((source, destination, content))

for source, destination, content in prepared:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        with destination.open("xb") as stream:
            stream.write(content)

    if destination.read_bytes() != content:
        raise RuntimeError("Copy verification failed.")

    print(
        destination.relative_to(ROOT),
        "| bytes:", len(content),
        "| SHA256:", sha(content)
    )

print("\nVerified report copies:", len(prepared))
print("Original results preserved; no staging, commit, or push.")

docs/research_reports/notebook06b_evidence_v1/calibration_v1.json | bytes: 1873 | SHA256: 87bfe4072cc4f8b1eb90fe961d4c348552f674223d6d49a85f5e7e0c7e1931e0
docs/research_reports/notebook06b_evidence_v1/weight_evaluation_v1.json | bytes: 3083 | SHA256: 1796dd817413f0746f8529aaefbac49f874d2226d451ae8759f54538dde69cfb
docs/research_reports/notebook06b_evidence_v1/paired_uncertainty_v1.json | bytes: 2904 | SHA256: cd41ade341243590f91f21105d0071712b65a7264a85b91a4ca0125787c21d0d
docs/research_reports/notebook06b_evidence_v1/end_to_end_decisions_v1.json | bytes: 18513 | SHA256: 89a9a3c3e2b4f8187089325a765b1a715b7059d40ed8446fe1f08e3881fab2c0
docs/research_reports/notebook07a_evidence_v1/mask_manifest_v1.json | bytes: 50418 | SHA256: bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a
docs/research_reports/notebook07a_evidence_v1/correctness_v1.json | bytes: 10128 | SHA256: b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206
docs/research_reports/notebook07a_evidence_

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def git(*args):
    result = subprocess.run(
        ["git", *args], cwd=ROOT,
        capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

files = [
    "notebooks/06b_detection_repair_pilot.ipynb",
    "notebooks/07a_invariant_bit_protection.ipynb",
    "src/calibration06b_v1.py",
    "src/decisions06b_e2e_v1.py",
    "src/end_to_end06b_v1.py",
    "src/evaluation06b_v1.py",
    "src/layer_analysis06b_v1.py",
    "src/report06b_e2e_v1.py",
    "src/invariant07a_extract_v1.py",
    "src/invariant07a_check_v1.py",
    "src/invariant07a_compare_v1.py",
    "src/invariant07a_timing_v1.py",
    "docs/research_reports/notebook06b_per_layer_v1.csv",
    "docs/research_reports/notebook06b_pilot_v1.md",
    "docs/research_reports/notebook07a_baseline_summary_v1.json",
]
for directory in (
    "experiments/notebook06b",
    "experiments/notebook07a",
    "docs/research_reports/notebook06b_evidence_v1",
    "docs/research_reports/notebook07a_evidence_v1",
    "docs/research_reports/notebook06b_end_to_end_v1",
):
    files.extend(
        str(path.relative_to(ROOT))
        for path in (ROOT / directory).rglob("*")
        if path.is_file()
    )

for number in (1600, 1604, 1650, 1700, 1704):
    files.append(
        f"docs/research_reports/notebook05b_screening_v1/progress_{number:05d}.json"
    )

files = sorted(set(files))
if git("diff", "--cached", "--name-only"):
    raise RuntimeError("Staging area is not empty; inspect before proceeding.")

tracked = set(git("ls-files").splitlines())

# Validate all selected files before staging anything.
for relative in files:
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(relative)
    if relative in tracked:
        raise RuntimeError(f"Selected file is already tracked: {relative}")
    if path.stat().st_size > 2 * 1024 * 1024:
        raise RuntimeError(f"Unexpectedly large file: {relative}")
    if path.suffix == ".py":
        compile(path.read_text(encoding="utf-8"), relative, "exec")
    elif path.suffix in (".json", ".ipynb"):
        payload = json.loads(path.read_bytes())
        if path.suffix == ".ipynb":
            if payload.get("nbformat") != 4:
                raise RuntimeError(f"Unexpected notebook format: {relative}")

git("add", "--", *files)

print("Selected files:", len(files))
print("Selected bytes:", sum((ROOT / name).stat().st_size for name in files))
print("\nStaged file status:")
print(git("diff", "--cached", "--name-status"))
print("\nStaged summary:")
print(git("diff", "--cached", "--stat"))
print("\nRemaining working-tree status:")
print(git("--no-optional-locks", "status", "--short"))
print("\nNo commit or push performed.")

Selected files: 34
Selected bytes: 657144

Staged file status:
A	docs/research_reports/notebook05b_screening_v1/progress_01600.json
A	docs/research_reports/notebook05b_screening_v1/progress_01604.json
A	docs/research_reports/notebook05b_screening_v1/progress_01650.json
A	docs/research_reports/notebook05b_screening_v1/progress_01700.json
A	docs/research_reports/notebook05b_screening_v1/progress_01704.json
A	docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv
A	docs/research_reports/notebook06b_end_to_end_v1/scenario_comparison_v1.csv
A	docs/research_reports/notebook06b_end_to_end_v1/summary_v1.json
A	docs/research_reports/notebook06b_evidence_v1/calibration_v1.json
A	docs/research_reports/notebook06b_evidence_v1/end_to_end_decisions_v1.json
A	docs/research_reports/notebook06b_evidence_v1/paired_uncertainty_v1.json
A	docs/research_reports/notebook06b_evidence_v1/weight_evaluation_v1.json
A	docs/research_reports/notebook06b_per_layer_v1.csv
A	docs/research_repor

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT, capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or result.stdout.strip()
            or f"Git command failed: {args[0]}"
        )
    return result.stdout.strip()

changes = git("diff", "--cached", "--name-status").splitlines()
if len(changes) != 34 or any(not row.startswith("A\t") for row in changes):
    raise RuntimeError("Staged set differs from the reviewed 34 new files.")

git("diff", "--cached", "--check")
print("Staged diff check: PASS")

print(git(
    "commit",
    "-m",
    "research: record 06b repair pilot, 07a invariant baseline and 05b progress",
))

print("\nCommit:", git("rev-parse", "HEAD"))
print("\nWorking-tree status:")
print(git("--no-optional-locks", "status", "--short") or "CLEAN")
print("\nLocal commit only; no push performed.")

RuntimeError: docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv:1: trailing whitespace.
+scenario_id,condition,state_index,q_final,AP50,AP50_95,AP75,AP_large,AP_medium,AP_small,AR1,AR10,AR100,AR_large,AR_medium,AR_small
docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv:2: trailing whitespace.
+S06B-E2E-001,shared_INT8_clean,0,7,0.6235200687849605,0.4842769592793853,0.5334446793894683,0.7057658898369708,0.4817085008041767,0.2747746377831853,0.40087117736099587,0.5929026768900507,0.6255526562312993,0.8256609326346168,0.6273432880883161,0.39721994235540353
docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv:3: trailing whitespace.
+S06B-E2E-001,faulty_no_intervention,1,6,0.6235196631699779,0.4842767447905447,0.5334445395956539,0.7057658219232268,0.4817085008041767,0.2747746618797954,0.40087117736099587,0.5929026768900507,0.6255526562312993,0.8256609326346168,0.6273432880883161,0.39721994235540353
docs/res

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

files = [
    "docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv",
    "docs/research_reports/notebook06b_end_to_end_v1/scenario_comparison_v1.csv",
    "docs/research_reports/notebook06b_per_layer_v1.csv",
]

for relative in files:
    content = (ROOT / relative).read_bytes()
    print(
        relative,
        "| CRLF:", content.count(b"\r\n"),
        "| LF:", content.count(b"\n"),
    )

result = subprocess.run(
    [
        "git", "-c",
        "core.whitespace=blank-at-eol,blank-at-eof,space-before-tab,cr-at-eol",
        "diff", "--cached", "--check",
    ],
    cwd=ROOT, capture_output=True, text=True, timeout=120,
)
print("\nCRLF-aware diff check return code:", result.returncode)
print(result.stdout.strip() or "(no warnings)")
if result.stderr.strip():
    print(result.stderr.strip())

print("\nNo file or persistent Git configuration changes; no commit or push.")

docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv | CRLF: 97 | LF: 97
docs/research_reports/notebook06b_end_to_end_v1/scenario_comparison_v1.csv | CRLF: 17 | LF: 17
docs/research_reports/notebook06b_per_layer_v1.csv | CRLF: 64 | LF: 64

CRLF-aware diff check return code: 0
(no warnings)

No file or persistent Git configuration changes; no commit or push.


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT, capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or result.stdout.strip()
            or "Git command failed."
        )
    return result.stdout.strip()

changes = git("diff", "--cached", "--name-status").splitlines()
if len(changes) != 34 or any(not row.startswith("A\t") for row in changes):
    raise RuntimeError("Staged set differs from the reviewed 34 new files.")

git(
    "-c",
    "core.whitespace=blank-at-eol,blank-at-eof,space-before-tab,cr-at-eol",
    "diff", "--cached", "--check",
)

print(git(
    "commit", "-m",
    "research: record 06b repair pilot, 07a invariant baseline and 05b progress",
))
print("\nCommit:", git("rev-parse", "HEAD"))
print("\nWorking-tree status:")
print(git("--no-optional-locks", "status", "--short") or "CLEAN")
print("\nLocal commit only; no push performed.")

[main 1155a56] research: record 06b repair pilot, 07a invariant baseline and 05b progress
 34 files changed, 8250 insertions(+)
 create mode 100644 docs/research_reports/notebook05b_screening_v1/progress_01600.json
 create mode 100644 docs/research_reports/notebook05b_screening_v1/progress_01604.json
 create mode 100644 docs/research_reports/notebook05b_screening_v1/progress_01650.json
 create mode 100644 docs/research_reports/notebook05b_screening_v1/progress_01700.json
 create mode 100644 docs/research_reports/notebook05b_screening_v1/progress_01704.json
 create mode 100644 docs/research_reports/notebook06b_end_to_end_v1/all_conditions_cocoeval_v1.csv
 create mode 100644 docs/research_reports/notebook06b_end_to_end_v1/scenario_comparison_v1.csv
 create mode 100644 docs/research_reports/notebook06b_end_to_end_v1/summary_v1.json
 create mode 100644 docs/research_reports/notebook06b_evidence_v1/calibration_v1.json
 create mode 100644 docs/research_reports/notebook06b_evidence_v1/end_to_

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit
import os
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
EXPECTED = "1155a56eae3accdfe3b4b820cdc3d539e9da958c"

def git(*args, timeout=120):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=timeout,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if result.returncode != 0:
        raise RuntimeError("Git operation failed; no force or automatic retry performed.")
    return result.stdout.strip()

if git("rev-parse", "HEAD") != EXPECTED:
    raise RuntimeError("Local HEAD differs from the reviewed commit.")
if git("branch", "--show-current") != "main":
    raise RuntimeError("Current branch is not main.")
if git("--no-optional-locks", "status", "--porcelain"):
    raise RuntimeError("Working tree is not clean.")

remote = git("remote", "get-url", "--push", "origin")
if remote.startswith("git@github.com:"):
    repository = remote.split(":", 1)[1]
elif remote.startswith(("https://", "ssh://")):
    parsed = urlsplit(remote)
    if parsed.hostname != "github.com":
        raise RuntimeError("Unexpected remote host.")
    repository = parsed.path.lstrip("/")
else:
    raise RuntimeError("Unexpected remote format.")

if repository.removesuffix(".git").rstrip("/") != (
    "kianaadineh/yolov8-fault-tolerance"
):
    raise RuntimeError("Unexpected remote repository.")

git("push", "origin", "main", timeout=180)
print("Push command: SUCCESS")

remote_line = git("ls-remote", "origin", "refs/heads/main")
remote_sha = remote_line.split()[0] if remote_line else None
print("Local commit:", EXPECTED)
print("Remote main:", remote_sha)

if remote_sha != EXPECTED:
    raise RuntimeError("Remote main does not match; publication requires review.")

print("Publication identity: PASS")

RuntimeError: Git operation failed; no force or automatic retry performed.

In [ ]:
from pathlib import Path
import os
import re
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def redact(text):
    text = re.sub(
        r"(https?://)[^/\s@]+@",
        r"\1[REDACTED]@",
        text,
    )
    text = re.sub(
        r"\b(?:github_pat_|gh[pousr]_)[A-Za-z0-9_]+",
        "[REDACTED_TOKEN]",
        text,
    )
    return text

commands = [
    ["git", "rev-parse", "HEAD"],
    ["git", "ls-remote", "origin", "refs/heads/main"],
]

for command in commands:
    print("\nCheck:", command[1])
    try:
        result = subprocess.run(
            command,
            cwd=ROOT,
            capture_output=True,
            text=True,
            timeout=60,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )
        print("Return code:", result.returncode)
        print("STDOUT:", redact(result.stdout.strip()) or "(empty)")
        print("STDERR:", redact(result.stderr.strip()) or "(empty)")
    except subprocess.TimeoutExpired:
        print("TIMEOUT")

print("\nRead-only diagnosis; no push or credential changes.")


Check: rev-parse
Return code: 0
STDOUT: 1155a56eae3accdfe3b4b820cdc3d539e9da958c
STDERR: (empty)

Check: ls-remote
Return code: 0
STDOUT: 97aca930e3c9fdb47e714e9d49decf4778d894c0	refs/heads/main
STDERR: (empty)

Read-only diagnosis; no push or credential changes.


In [ ]:
from pathlib import Path
import os
import re
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def redact(text):
    text = re.sub(r"(https?://)[^/\s@]+@", r"\1[REDACTED]@", text)
    return re.sub(
        r"\b(?:github_pat_|gh[pousr]_)[A-Za-z0-9_]+",
        "[REDACTED_TOKEN]",
        text,
    )

try:
    result = subprocess.run(
        ["git", "push", "--dry-run", "origin", "main"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=60,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    print("Return code:", result.returncode)
    print("STDOUT:", redact(result.stdout.strip()) or "(empty)")
    print("STDERR:", redact(result.stderr.strip()) or "(empty)")
except subprocess.TimeoutExpired:
    print("TIMEOUT")

print("\nDry run only; no publication or credential changes.")

Return code: 128
STDOUT: (empty)
STDERR: fatal: could not read Username for 'https://github.com': terminal prompts disabled

Dry run only; no publication or credential changes.


In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import os
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
EXPECTED = "1155a56eae3accdfe3b4b820cdc3d539e9da958c"

def publish():
    def run(args, env=None):
        return subprocess.run(
            ["git", *args], cwd=ROOT, env=env,
            capture_output=True, text=True, timeout=180,
        )

    head = run(["rev-parse", "HEAD"])
    if head.returncode or head.stdout.strip() != EXPECTED:
        raise RuntimeError("Unexpected local commit.")

    remote = run(["remote", "get-url", "--push", "origin"])
    if remote.returncode or remote.stdout.strip().rstrip("/") not in (
        "https://github.com/kianaadineh/yolov8-fault-tolerance",
        "https://github.com/kianaadineh/yolov8-fault-tolerance.git",
    ):
        raise RuntimeError("Unexpected push URL; nothing published.")

    token = getpass("GitHub token (hidden; do not paste into chat): ").strip()
    if not token:
        raise RuntimeError("No token entered.")

    encoded = base64.b64encode(
        ("kianaadineh:" + token).encode()
    ).decode()

    env = os.environ.copy()
    env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "2",
        "GIT_CONFIG_KEY_0": "credential.helper",
        "GIT_CONFIG_VALUE_0": "",
        "GIT_CONFIG_KEY_1": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_1": "Authorization: Basic " + encoded,
    })

    try:
        result = run(["push", "origin", "main"], env)
        if result.returncode:
            message = result.stderr.replace(token, "[REDACTED]")
            message = message.replace(encoded, "[REDACTED]")
            print(message.strip())
            raise RuntimeError("Push failed; no force or automatic retry.")

        check = run(["ls-remote", "origin", "refs/heads/main"], env)
        remote_sha = (
            check.stdout.split()[0]
            if check.returncode == 0 and check.stdout.strip() else None
        )
        print("Local commit:", EXPECTED)
        print("Remote main:", remote_sha)
        if remote_sha != EXPECTED:
            raise RuntimeError("Publication identity not confirmed.")
        print("Publication identity: PASS")
    finally:
        env.clear()
        token = encoded = None

publish()

GitHub token (hidden; do not paste into chat): ··········
fatal: could not read Username for 'https://github.com': terminal prompts disabled


RuntimeError: Push failed; no force or automatic retry.

In [ ]:
from getpass import getpass
from pathlib import Path
import base64
import os
import subprocess

def push_with_access():
    root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
    expected = "1155a56eae3accdfe3b4b820cdc3d539e9da958c"
    url = "https://github.com/kianaadineh/yolov8-fault-tolerance.git"

    def git(args, env=None):
        return subprocess.run(
            ["git", *args], cwd=root, env=env,
            capture_output=True, text=True, timeout=180,
        )

    head = git(["rev-parse", "HEAD"])
    if head.returncode or head.stdout.strip() != expected:
        raise RuntimeError("Local commit differs; nothing pushed.")

    token = getpass("GitHub token — hidden input: ").strip()
    if not token:
        raise RuntimeError("No token entered.")

    authorization = base64.b64encode(
        f"kianaadineh:{token}".encode()
    ).decode()

    env = os.environ.copy()
    env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "2",
        "GIT_CONFIG_KEY_0": "credential.helper",
        "GIT_CONFIG_VALUE_0": "",
        "GIT_CONFIG_KEY_1": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_1": f"Authorization: Basic {authorization}",
    })

    try:
        result = git(["push", url, f"{expected}:refs/heads/main"], env)
        if result.returncode:
            message = result.stderr.replace(token, "[REDACTED]")
            message = message.replace(authorization, "[REDACTED]")
            print(message.strip())
            raise RuntimeError("Push failed; no force used.")

        check = git(["ls-remote", url, "refs/heads/main"], env)
        remote_sha = (
            check.stdout.split()[0]
            if check.returncode == 0 and check.stdout.strip() else None
        )
        print("Remote main:", remote_sha)
        if remote_sha != expected:
            raise RuntimeError("Publication not confirmed.")
        print("Publication identity: PASS")
    finally:
        env.clear()
        token = authorization = None

push_with_access()

GitHub token — hidden input: ··········
Remote main: 1155a56eae3accdfe3b4b820cdc3d539e9da958c
Publication identity: PASS


In [1]:
from google.colab import drive
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys
import torch

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
torch.set_num_threads(1)

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("Repository exists:", ROOT.is_dir())

for package in ("torch", "numpy", "ultralytics"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

if ROOT.is_dir():
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=ROOT, capture_output=True, text=True, timeout=30,
    )
    print("Git HEAD:", result.stdout.strip() or result.stderr.strip())

    for relative in (
        "src/invariant07a_extract_v1.py",
        "src/invariant07a_check_v1.py",
        "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json",
        "results/notebook07a/invariant_masks_v1/manifest.json",
    ):
        print(relative, "FOUND" if (ROOT / relative).is_file() else "MISSING")

print("No installation, model loading, or experiment execution.")

Mounted at /content/drive
Python: 3.13.16
CUDA available: True
Repository exists: True
torch: 2.14.1+cu130
numpy: 2.4.6
ultralytics: NOT INSTALLED
Git HEAD: 1155a56eae3accdfe3b4b820cdc3d539e9da958c
src/invariant07a_extract_v1.py FOUND
src/invariant07a_check_v1.py FOUND
experiments/notebook07a/invariant_bit_baseline_protocol_v1.json FOUND
results/notebook07a/invariant_masks_v1/manifest.json FOUND
No installation, model loading, or experiment execution.


In [2]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    capture_output=True, text=True, timeout=30,
)
print("GPU:", result.stdout.strip() or result.stderr.strip())

for package in (
    "torch", "torchvision", "ultralytics", "numpy",
    "pycocotools", "opencv-python", "Pillow"
):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

protocol = (
    ROOT / "experiments/notebook07a/group_bit_design_comparison_protocol_v1.json"
)
print("New comparison protocol exists:", protocol.is_file())
print("No installation, inference, or campaign execution.")

GPU: Tesla T4, 15360 MiB, 14910 MiB
torch: 2.14.1+cu130
torchvision: 0.29.1+cu130
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
opencv-python: 5.0.0.93
Pillow: 11.3.0
New comparison protocol exists: False
No installation, inference, or campaign execution.


In [3]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

protected = (
    "torch", "torchvision", "numpy",
    "pycocotools", "opencv-python", "Pillow"
)
before = {name: version(name) for name in protected}

try:
    installed = version("ultralytics")
except PackageNotFoundError:
    installed = None

if installed is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps", "ultralytics==8.4.189",
        ],
        check=True,
    )
elif installed != "8.4.189":
    raise RuntimeError(
        f"Unexpected Ultralytics version: {installed}; nothing changed."
    )

after = {name: version(name) for name in protected}
if before != after:
    raise RuntimeError("Protected package versions changed.")

from ultralytics import YOLO

print("Ultralytics:", version("ultralytics"))
print("Protected versions unchanged:", after)
print("YOLO import: PASS")
print("No model loading, inference, or campaign execution.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
Protected versions unchanged: {'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'numpy': '2.4.6', 'pycocotools': '2.0.11', 'opencv-python': '5.0.0.93', 'Pillow': '11.3.0'}
YOLO import: PASS
No model loading, inference, or campaign execution.


In [4]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import importlib.util
import json
import platform
import torch

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()

expected_plan_sha = json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(plan_bytes).hexdigest() != expected_plan_sha:
    raise RuntimeError("Plan checksum mismatch.")

plan = json.loads(plan_bytes)
source_path = ROOT / "src/screen05b_campaign_v1.py"
if hashlib.sha256(source_path.read_bytes()).hexdigest() != plan["source_sha256"]:
    raise RuntimeError("Campaign source checksum mismatch.")

spec = importlib.util.spec_from_file_location("screen05b_resume", source_path)
screen05b_resume = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen05b_resume)

torch.set_num_threads(1)
campaign05b = screen05b_resume.Campaign(ROOT.parent)
campaign05b._load_sources()
campaign05b._model("cuda:0")
campaign05b._load_plan()
campaign05b._cache()

previous = campaign05b.base.read_record(
    campaign05b.out / "session.json.gz"
)
environment = {
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in (
            "torch", "torchvision", "ultralytics", "numpy",
            "pycocotools", "opencv-python", "Pillow"
        )
    },
    "gpu": torch.cuda.get_device_name(0),
    "torch_threads": torch.get_num_threads(),
    "torch_cuda_build": torch.version.cuda,
    "tf32": False,
    "deterministic_algorithms": True,
}
current = {
    "plan_sha256": campaign05b.plan_sha,
    "source_sha256": campaign05b.source_sha,
    "environment": environment,
    "image_sha256": campaign05b.image_hashes,
    "clean_state": campaign05b.clean_hash,
}

for key in current:
    print(key, "MATCH" if previous.get(key) == current[key] else "DIFFERENT")

if previous.get("environment") != environment:
    print("\nPrevious environment:", previous.get("environment"))
    print("Current environment:", environment)

partials = list(campaign05b.out.rglob("*.part"))
failures = list(campaign05b.out.glob("*_failure.json.gz"))
evaluations = list(campaign05b.out.glob("S05B-*_evaluation.json.gz"))

print("\nSaved scenario evaluations:", len(evaluations))
print("Partial files:", len(partials))
print("Failure records:", len(failures))
for path in partials + failures:
    print(path.relative_to(campaign05b.out))

resume05b_ok = previous == current and not partials and not failures
print("Resume compatibility:", "PASS" if resume05b_ok else "REVIEW REQUIRED")
print("No new scenario inference or result writes.")

plan_sha256 MATCH
source_sha256 MATCH
environment MATCH
image_sha256 MATCH
clean_state MATCH

Saved scenario evaluations: 1704
Partial files: 0
Failure records: 0
Resume compatibility: PASS
No new scenario inference or result writes.


In [1]:
if not resume05b_ok:
    raise RuntimeError("Resume compatibility has not passed.")

progress05b = campaign05b.run(
    max_new_scenarios=None,
    max_wall_seconds=24 * 60 * 60,
)

NameError: name 'resume05b_ok' is not defined

In [2]:
from google.colab import drive
from pathlib import Path
import gzip
import hashlib
import json
import torch

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
OUT = ROOT / "results/notebook05b/int8_screening_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    with gzip.open(path, "rb") as stream:
        envelope = json.load(stream)
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {path.name}")
    return payload

print("CUDA available:", torch.cuda.is_available())
print("Campaign directory exists:", OUT.is_dir())
if not OUT.is_dir():
    raise FileNotFoundError("Campaign results directory not found.")

session = read_record(OUT / "session.json.gz")
fingerprint = hashlib.sha256(canonical(session)).hexdigest()

plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()
if hashlib.sha256(plan_bytes).hexdigest() != session["plan_sha256"]:
    raise RuntimeError("Plan differs from saved campaign session.")
plan = json.loads(plan_bytes)

evaluations = {
    path.name.removesuffix("_evaluation.json.gz"): path
    for path in OUT.glob("S05B-*_evaluation.json.gz")
}
predictions = {
    path.name.removesuffix("_predictions.json.gz"): path
    for path in OUT.glob("S05B-*_predictions.json.gz")
}
partials = sorted(OUT.rglob("*.part"))
failures = sorted(OUT.glob("*_failure.json.gz"))

valid = set()
issues = []

for scenario in plan["scenarios"]:
    sid = scenario["scenario_id"]
    path = evaluations.get(sid)
    if path is None:
        continue
    try:
        record = read_record(path)
        if (
            record["kind"] != "scenario_evaluation"
            or record["session_fingerprint"] != fingerprint
            or record["plan_sha256"] != session["plan_sha256"]
            or record["scenario"] != scenario
        ):
            raise RuntimeError("Evaluation provenance mismatch.")
        if sid not in predictions:
            raise RuntimeError("Prediction archive missing.")
        valid.add(sid)
    except Exception as exc:
        issues.append((sid, str(exc)))

planned = {row["scenario_id"] for row in plan["scenarios"]}
unexpected = (set(evaluations) | set(predictions)) - planned
prediction_only = sorted(set(predictions) - set(evaluations))

prefix = 0
for scenario in plan["scenarios"]:
    if scenario["scenario_id"] not in valid:
        break
    prefix += 1

total = len(plan["scenarios"])
print("\nVerified evaluation records:", len(valid), "/", total)
print("Progress, %:", round(100 * len(valid) / total, 2))
print("Remaining:", total - len(valid))
print("Consecutive completed prefix:", prefix)
print("Prediction archives without evaluation:", len(prediction_only))
print("Partial files:", len(partials))
print("Failure records:", len(failures))
print("Invalid evaluation records:", len(issues))
print("Unexpected scenario IDs:", len(unexpected))

for sid in prediction_only[:10]:
    print("PREDICTION ONLY:", sid)
for path in partials + failures:
    print("REQUIRES REVIEW:", path.relative_to(OUT))
for sid, message in issues[:10]:
    print("INVALID:", sid, message)
for sid in sorted(unexpected)[:10]:
    print("UNEXPECTED:", sid)

for path in failures[-3:]:
    try:
        record = read_record(path)
        print("\nFailure:", path.name)
        for key in ("exception_type", "message", "clean_state_restored"):
            print(f"{key}:", record.get(key))
    except Exception as exc:
        print("Failure record unreadable:", path.name, str(exc))

if prefix < total:
    print("\nNext scenario in plan:", plan["scenarios"][prefix]["scenario_id"])

print("\nEvaluation payloads checked; prediction archive hashes not rechecked.")
print("No inference, file deletion, modification, or campaign execution.")

Mounted at /content/drive
CUDA available: False
Campaign directory exists: True

Verified evaluation records: 2198 / 5040
Progress, %: 43.61
Remaining: 2842
Consecutive completed prefix: 2198
Prediction archives without evaluation: 1
Partial files: 0
Failure records: 0
Invalid evaluation records: 0
Unexpected scenario IDs: 0
PREDICTION ONLY: S05B-02199

Next scenario in plan: S05B-02199

Evaluation payloads checked; prediction archive hashes not rechecked.
No inference, file deletion, modification, or campaign execution.


In [3]:
from pathlib import Path
import gzip
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
OUT = ROOT / "results/notebook05b/int8_screening_v1"
SID = "S05B-02199"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    content = path.read_bytes()
    envelope = json.loads(gzip.decompress(content))
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {path.name}")
    return payload, hashlib.sha256(content).hexdigest()

session, _ = read_record(OUT / "session.json.gz")
fingerprint = hashlib.sha256(canonical(session)).hexdigest()

plan_bytes = (
    ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
).read_bytes()
if hashlib.sha256(plan_bytes).hexdigest() != session["plan_sha256"]:
    raise RuntimeError("Plan checksum mismatch.")
plan = json.loads(plan_bytes)
scenario = next(row for row in plan["scenarios"] if row["scenario_id"] == SID)

record, archive_sha = read_record(OUT / f"{SID}_predictions.json.gz")
if (
    record["kind"] != "predictions"
    or record["session_fingerprint"] != fingerprint
    or record["plan_sha256"] != session["plan_sha256"]
    or record["scenario"] != scenario
):
    raise RuntimeError("Prediction provenance mismatch.")

data = record["data"]
if (
    data["image_ids"] != plan["image_ids"]
    or [row["image_id"] for row in data["per_image"]] != plan["image_ids"]
):
    raise RuntimeError("Image execution record mismatch.")

print("Prediction archive integrity and provenance: PASS")
print("Scenario:", json.dumps(scenario, indent=2))
print("Images:", len(data["image_ids"]))
print("Predictions:", len(data["predictions"]))
print("Recorded model-state SHA256:", record["model_state_sha256"])
print("Prediction archive SHA256:", archive_sha)
print("Pipeline seconds:", data["pipeline_seconds"])
print("Evaluation already exists:", (OUT / f"{SID}_evaluation.json.gz").exists())
print("No inference or file changes.")

Prediction archive integrity and provenance: PASS
Scenario: {
  "layer": "model.2.cv2.conv",
  "bit": 2,
  "replicate": 3,
  "flat_index": 2929,
  "scenario_id": "S05B-02199",
  "q_clean": -5,
  "q_faulty": -1,
  "scale": 0.0018685408867895603,
  "clean_weight": -0.009342703968286514,
  "faulty_weight": -0.0018685408867895603
}
Images: 200
Predictions: 19827
Recorded model-state SHA256: 50ff9a5038efbee2d4e7edb8313ea2dc2296cccda6b8009e36c935f730ce423d
Prediction archive SHA256: 359ee3cc6d62d94ea594e25a36ce248d20a0dde7ed42fdf2d58836289a7f4d69
Pipeline seconds: 8.898366462999547
Evaluation already exists: False
No inference or file changes.


In [4]:
from pathlib import Path
from importlib.metadata import version
import contextlib
import hashlib
import importlib.util
import io
import json
import math
import platform

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
OUT = ROOT / "results/notebook05b/int8_screening_v1"
SID = "S05B-02199"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    import gzip
    envelope = json.loads(gzip.decompress(path.read_bytes()))
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {path.name}")
    return envelope["payload"]

session = read_record(OUT / "session.json.gz")
source_path = ROOT / "src/screen05b_campaign_v1.py"
if sha(source_path.read_bytes()) != session["source_sha256"]:
    raise RuntimeError("Campaign source checksum mismatch.")

spec = importlib.util.spec_from_file_location("screen05b_cpu_replay", source_path)
screen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen)

# Source loading does not construct a model or run inference.
campaign = screen.Campaign(ROOT.parent)
campaign._load_sources()

plan_bytes = campaign.plan_path.read_bytes()
if sha(plan_bytes) != session["plan_sha256"]:
    raise RuntimeError("Plan checksum mismatch.")
plan = json.loads(plan_bytes)
scenario = next(row for row in plan["scenarios"] if row["scenario_id"] == SID)

fingerprint = sha(canonical(session))
campaign.category_ids = {row["id"] for row in campaign.categories}

prediction_path = OUT / f"{SID}_predictions.json.gz"
prediction_bytes = prediction_path.read_bytes()
if sha(prediction_bytes) != (
    "359ee3cc6d62d94ea594e25a36ce248d20a0dde7ed42fdf2d58836289a7f4d69"
):
    raise RuntimeError("Prediction archive changed.")

prediction = campaign.base.read_record(prediction_path)
baseline = campaign.base.read_record(OUT / "clean_evaluation.json.gz")

for saved, kind in ((prediction, "predictions"), (baseline, "baseline")):
    if (
        saved["kind"] != kind
        or saved["session_fingerprint"] != fingerprint
        or saved["plan_sha256"] != session["plan_sha256"]
    ):
        raise RuntimeError("Stored record provenance mismatch.")

if prediction["scenario"] != scenario:
    raise RuntimeError("Scenario mismatch.")
if baseline["prediction_sha256"] != sha(
    (OUT / "clean_predictions.json.gz").read_bytes()
):
    raise RuntimeError("Clean prediction archive checksum mismatch.")
if version("pycocotools") != session["environment"]["packages"]["pycocotools"]:
    raise RuntimeError("COCOeval package version mismatch.")

campaign._validate_predictions(prediction["data"])

from pycocotools.coco import COCO
with contextlib.redirect_stdout(io.StringIO()):
    campaign.gt = COCO(str(campaign.annotation_path))

metrics, seconds = campaign._evaluate(prediction["data"]["predictions"])
if any(not math.isfinite(value) for value in metrics.values()) or metrics["AP50_95"] < 0:
    raise RuntimeError("Invalid evaluation metrics.")

change = metrics["AP50_95"] - baseline["metrics"]["AP50_95"]
payload = {
    "status": "CPU_COCOEVAL_REPLAY_NOT_OFFICIAL_CAMPAIGN_COMPLETION",
    "scenario": scenario,
    "source_session_fingerprint": fingerprint,
    "source_runner_sha256": session["source_sha256"],
    "prediction_sha256": sha(prediction_bytes),
    "recorded_fault_model_state_sha256": prediction["model_state_sha256"],
    "annotation_sha256": campaign.config["annotation_sha256"],
    "image_ids": campaign.ids,
    "metrics": metrics,
    "saved_clean_metrics": baseline["metrics"],
    "AP50_95_change": change,
    "nonnegative_AP50_95_loss": max(0.0, -change),
    "evaluation_seconds": seconds,
    "environment": {
        "python": platform.python_version(),
        "pycocotools": version("pycocotools"),
        "device": "cpu"
    },
    "limitations": [
        "Replays saved GPU predictions; no new model inference.",
        "Model-state identity and exact restoration were not reverified on a reconstructed model.",
        "Original campaign results and completion count are unchanged."
    ]
}

destination = (
    ROOT / "docs/research_reports/notebook05b_cpu_replay_v1"
    / f"{SID}_replay_v1.json"
)
content = (
    json.dumps(
        {"payload_sha256": sha(canonical(payload)), "payload": payload},
        indent=2, allow_nan=False
    ) + "\n"
).encode("utf-8")

if destination.exists():
    raise FileExistsError("Replay report exists; nothing overwritten.")
destination.parent.mkdir(parents=True, exist_ok=True)
with destination.open("xb") as stream:
    stream.write(content)
assert destination.read_bytes() == content

print("COCOeval metrics:", json.dumps(metrics, indent=2))
print("AP50:95 change, percentage points:", 100 * change)
print("Saved:", destination)
print("SHA256:", sha(content))
print("No inference or modifications to original campaign records.")

COCOeval metrics: {
  "AP50_95": 0.48421833467401065,
  "AP50": 0.6233213596062622,
  "AP75": 0.5331010930663347,
  "AP_small": 0.27481409984097727,
  "AP_medium": 0.48163054562795327,
  "AP_large": 0.7057968488704993,
  "AR1": 0.40089467360159736,
  "AR10": 0.592977200753861,
  "AR100": 0.6252868182827942,
  "AR_small": 0.3960888170002685,
  "AR_medium": 0.6272473392610239,
  "AR_large": 0.8256609326346168
}
AP50:95 change, percentage points: -0.005862460537464642
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook05b_cpu_replay_v1/S05B-02199_replay_v1.json
SHA256: d15bc5d6e7050246e6f6a24b9934d31339cd3662ced75e663e039dc8c4c3f985
No inference or modifications to original campaign records.


In [5]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
reference_path = (
    ROOT / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json"
)
reference_bytes = reference_path.read_bytes()

if hashlib.sha256(reference_bytes).hexdigest() != (
    "74785069999784731bcc2917a5fd44fa8b698c102f76a9043b1f022b8303a799"
):
    raise RuntimeError("Reference protocol checksum mismatch.")

reference = json.loads(reference_bytes)
protocol = {
    "version": "1.0",
    "status": "FROZEN_CPU_DESIGN_COMPARISON_BEFORE_EXTRACTION",
    "reference_protocol_sha256": hashlib.sha256(reference_bytes).hexdigest(),
    "checkpoint_sha256": reference["checkpoint_sha256"],
    "clean_model_state_sha256": reference["clean_model_state_sha256"],
    "source_screening_plan_sha256": reference["source_screening_plan_sha256"],
    "candidates": [
        {"name": "reference_g9_b456", "group_size": 9, "bits": [4, 5, 6]},
        {"name": "g3_b4567", "group_size": 3, "bits": [4, 5, 6, 7]},
        {"name": "g9_b4567", "group_size": 9, "bits": [4, 5, 6, 7]},
        {"name": "g27_b4567", "group_size": 27, "bits": [4, 5, 6, 7]},
    ],
    "grouping": {
        "order": "contiguous_flattened_weights_within_each_output_channel",
        "cross_output_channel_groups": False,
        "tails": "leave_incomplete_groups_unprotected",
        "reference_g9_must_match_existing_07a_masks": True
    },
    "storage": {
        "mask_bytes_per_group": 1,
        "encoding": "low_n_bits_force_zero_next_n_bits_force_one",
        "include_zero_masks": True,
        "count_serialized_descriptors_and_metadata": True,
        "compression_not_used_as_runtime_memory_estimate": True
    },
    "metrics": [
        "protected_positions_by_bit_and_layer",
        "all_8_bit_coverage_including_tails",
        "raw_mask_bytes",
        "descriptor_bytes",
        "protected_positions_per_mask_plus_descriptor_byte"
    ],
    "selection": {
        "maximum_nominees_for_next_AP_pilot": 1,
        "engineering_storage_budget_fraction": 0.12,
        "budget_denominator": "all_eligible_logical_INT8_weight_bytes",
        "budget_numerator": "raw_mask_bytes_plus_serialized_descriptor_bytes",
        "rule": (
            "Among candidates within budget, choose maximum protected "
            "weight_bit_positions per budget byte; ties favor smaller storage, "
            "then smaller group size."
        ),
        "if_none_qualifies": "report_no_candidate_without_relaxing_budget",
        "meaning": "coverage_cost_nomination_only_not_AP_optimality",
        "uses_06b_recovery_or_AP_for_selection": False
    },
    "limitations": [
        "Engineering screening, not novelty or protection efficacy confirmation.",
        "Uniform weight-bit coverage is not the measured 05b layer-bit risk.",
        "Trusted masks, metadata, scales and checker.",
        "Runtime RAM and scan latency require separate measurements.",
        "No AP inference or threshold tuning in this comparison."
    ],
    "existing_protocols_changed": False
}

destination = (
    ROOT / "experiments/notebook07a/group_bit_design_comparison_protocol_v1.json"
)
content = (
    json.dumps(protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; nothing overwritten.")
    print("IDENTICAL PROTOCOL ALREADY EXISTS")
else:
    with destination.open("xb") as stream:
        stream.write(content)
    print("NEW PROTOCOL SAVED")

assert destination.read_bytes() == content
print("Path:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No installation, model loading, or experiment execution.")

NEW PROTOCOL SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook07a/group_bit_design_comparison_protocol_v1.json
SHA256: 9ca97fde22132dbb403c586f69d6cc24906f223cac6113e74108e46157325c43
No installation, model loading, or experiment execution.


In [6]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

protected = ("torch", "numpy")
before = {name: version(name) for name in protected}

try:
    installed = version("ultralytics")
except PackageNotFoundError:
    installed = None

if installed is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps", "ultralytics==8.4.189",
        ],
        check=True,
    )
elif installed != "8.4.189":
    raise RuntimeError(f"Unexpected Ultralytics version: {installed}")

from ultralytics import YOLO
import torch

after = {name: version(name) for name in protected}
if before != after:
    raise RuntimeError("Protected package versions changed.")

print("Ultralytics:", version("ultralytics"))
print("Protected versions unchanged:", after)
print("CUDA available:", torch.cuda.is_available())
print("Required import: PASS")
print("No model loading or experiment execution.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
Protected versions unchanged: {'torch': '2.14.1+cpu', 'numpy': '2.4.6'}
CUDA available: False
Required import: PASS
No model loading or experiment execution.


In [7]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/group_bit07a_compare_v1.py"

source = r'''"""Frozen CPU comparison of four invariant-mask designs; no COCO inference."""
import csv
import hashlib
import importlib.util
import io
import json
import os
import platform
import time
from pathlib import Path

PROTOCOL_SHA = "9ca97fde22132dbb403c586f69d6cc24906f223cac6113e74108e46157325c43"
REFERENCE_MANIFEST_SHA = "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record payload checksum mismatch.")
    return envelope["payload"]

def publish(path, content):
    partial = path.with_name(path.name + ".part")
    if path.exists() or partial.exists():
        raise FileExistsError(f"Existing output requires review: {path.name}")
    with partial.open("xb") as stream:
        stream.write(content)
        stream.flush()
        os.fsync(stream.fileno())
    if partial.read_bytes() != content:
        raise RuntimeError("Write verification failed.")
    partial.rename(path)

def run(repo):
    import numpy as np
    import torch
    from importlib.metadata import version

    repo = Path(repo)
    out = repo / "results/notebook07a/group_bit_design_comparison_v1"
    if out.exists():
        raise FileExistsError("Comparison directory exists; inspect before rerunning.")

    protocol = json.loads(verified(
        repo / "experiments/notebook07a/group_bit_design_comparison_protocol_v1.json",
        PROTOCOL_SHA
    ))
    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    reference_folder = repo / "results/notebook07a/invariant_masks_v1"
    reference = unpack(verified(
        reference_folder / "manifest.json", REFERENCE_MANIFEST_SHA
    ))
    reference_bytes = verified(
        reference_folder / "masks.npz", reference["mask_archive_sha256"]
    )
    with np.load(io.BytesIO(reference_bytes), allow_pickle=False) as archive:
        reference_masks = {key: archive[key].copy() for key in archive.files}

    source_path = repo / "src/screen05b_campaign_v1.py"
    verified(source_path, plan["source_sha256"])
    spec = importlib.util.spec_from_file_location("screen07a_design", source_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if (
        prepared.clean_hash != protocol["clean_model_state_sha256"]
        or prepared.clean_hash != reference["clean_model_state_sha256"]
    ):
        raise RuntimeError("Clean model identity mismatch.")

    reference_by_name = {row["layer"]: row for row in reference["inventory"]}
    weights = {}
    for item in plan["inventory"]:
        name = item["name"]
        q = prepared.quantized[name][0].cpu().contiguous().numpy().copy()
        previous = reference_by_name[name]
        if (
            list(q.shape) != item["weight_shape"]
            or q.dtype != np.int8
            or sha(q.tobytes()) != previous["clean_integer_array_sha256"]
        ):
            raise RuntimeError("Clean integer tensor mismatch.")
        weights[name] = q
    total_weights = sum(q.size for q in weights.values())
    popcount = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)

    rows, layer_rows, outputs = [], [], []
    for candidate in protocol["candidates"]:
        started = time.perf_counter()
        group_size = candidate["group_size"]
        bits = candidate["bits"]
        nbits = len(bits)
        arrays, descriptors = {}, []
        protected_total = groups_total = tails_total = 0
        by_bit = {str(bit): 0 for bit in range(8)}

        for layer_id, item in enumerate(plan["inventory"]):
            name = item["name"]
            q = weights[name]
            o = q.shape[0]
            width = q.size // o
            complete = (width // group_size) * group_size
            blocks = q.view(np.uint8).reshape(o, width)[:, :complete]
            blocks = blocks.reshape(-1, group_size)

            all_one = np.bitwise_and.reduce(blocks, axis=1)
            all_zero = np.bitwise_not(np.bitwise_or.reduce(blocks, axis=1))
            packed = np.zeros(len(blocks), dtype=np.uint8)
            counts = {str(bit): 0 for bit in range(8)}

            for offset, bit in enumerate(bits):
                zero = ((all_zero >> bit) & 1).astype(np.uint8)
                one = ((all_one >> bit) & 1).astype(np.uint8)
                if np.any(zero & one):
                    raise RuntimeError("Conflicting invariant flags.")
                packed |= zero << offset
                packed |= one << (offset + nbits)
                counts[str(bit)] = int(np.count_nonzero(zero | one)) * group_size
                by_bit[str(bit)] += counts[str(bit)]

            key = f"layer_{layer_id:03d}"
            if candidate["name"] == "reference_g9_b456":
                old_key = reference_by_name[name]["array_key"]
                if not np.array_equal(packed, reference_masks[old_key]):
                    raise RuntimeError(f"Reference mask reproduction failed: {name}")

            protected = int(popcount[packed].sum()) * group_size
            if protected != sum(counts.values()):
                raise RuntimeError("Invariant coverage accounting mismatch.")
            tails = o * (width - complete)
            if len(blocks) * group_size + tails != q.size:
                raise RuntimeError("Grouping accounting mismatch.")

            arrays[key] = packed
            protected_total += protected
            groups_total += len(blocks)
            tails_total += tails
            descriptors.append({
                "layer": name,
                "shape": list(q.shape),
                "array_key": key,
                "groups": len(blocks),
                "mask_array_sha256": sha(packed.tobytes())
            })
            layer_rows.append({
                "candidate": candidate["name"],
                "layer": name,
                "weights": int(q.size),
                "groups": len(blocks),
                "tail_weights": tails,
                "protected_positions": protected,
                **{f"protected_bit_{bit}": counts[str(bit)] for bit in range(8)}
            })

        buffer = io.BytesIO()
        np.savez_compressed(buffer, **arrays)
        archive_bytes = buffer.getvalue()
        with np.load(io.BytesIO(archive_bytes), allow_pickle=False) as archive:
            if set(archive.files) != set(arrays):
                raise RuntimeError("Archive schema mismatch.")
            for key in arrays:
                if not np.array_equal(arrays[key], archive[key]):
                    raise RuntimeError("Archive round-trip mismatch.")

        descriptor = {
            "version": "1.0",
            "protocol_sha256": PROTOCOL_SHA,
            "candidate": candidate,
            "representation": "logical_signed_INT8",
            "grouping": protocol["grouping"],
            "encoding": "low_n_bits_force_zero_next_n_bits_force_one",
            "mask_dtype": "uint8",
            "mask_archive": candidate["name"] + "_masks.npz",
            "mask_archive_sha256": sha(archive_bytes),
            "reference_metadata_trusted": True,
            "layers": descriptors
        }
        descriptor_bytes = canonical(descriptor) + b"\n"
        mask_bytes = sum(a.nbytes for a in arrays.values())
        cost = mask_bytes + len(descriptor_bytes)
        overhead = cost / total_weights
        row = {
            "candidate": candidate["name"],
            "group_size": group_size,
            "candidate_bits": ",".join(map(str, bits)),
            "weights": total_weights,
            "groups": groups_total,
            "tail_weights": tails_total,
            "protected_positions": protected_total,
            "all_bit_coverage_pct": 100 * protected_total / (8 * total_weights),
            "raw_mask_bytes": mask_bytes,
            "descriptor_bytes": len(descriptor_bytes),
            "budget_bytes": cost,
            "storage_overhead_pct": 100 * overhead,
            "protected_positions_per_budget_byte": protected_total / cost,
            "within_budget": overhead <= protocol["selection"]["engineering_storage_budget_fraction"],
            "compressed_archive_bytes": len(archive_bytes),
            "construction_and_serialization_seconds": time.perf_counter() - started,
            **{f"protected_bit_{bit}": by_bit[str(bit)] for bit in range(8)}
        }
        rows.append(row)
        outputs.extend([
            (candidate["name"] + "_masks.npz", archive_bytes),
            (candidate["name"] + "_descriptor.json", descriptor_bytes)
        ])

    eligible = [row for row in rows if row["within_budget"]]
    eligible.sort(key=lambda row: (
        -row["protected_positions_per_budget_byte"],
        row["budget_bytes"], row["group_size"], row["candidate"]
    ))
    nominee = eligible[0]["candidate"] if eligible else None
    prepared._assert_state(prepared.clean_hash)

    payload = {
        "status": "CPU_COVERAGE_COST_SCREEN_COMPLETE_NOT_AP_EVALUATION",
        "protocol_sha256": PROTOCOL_SHA,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "reference_manifest_sha256": REFERENCE_MANIFEST_SHA,
        "clean_model_state_sha256": prepared.clean_hash,
        "reference_masks_reproduced_exactly": True,
        "environment": {
            "python": platform.python_version(),
            "packages": {p: version(p) for p in ("numpy", "torch", "ultralytics")},
            "device": "cpu", "torch_threads": torch.get_num_threads()
        },
        "budget_fraction": protocol["selection"]["engineering_storage_budget_fraction"],
        "rows": rows,
        "nominee": nominee,
        "cost_definition": (
            "Raw mask arrays plus exact serialized runtime descriptor bytes. "
            "Compressed archives and audit-only result tables are reported separately."
        ),
        "limitations": protocol["limitations"] + [
            "New candidate correction functions and scan timings are not yet validated.",
            "Selection maximizes uniform protected-position efficiency, not AP benefit.",
            "No sensitivity-informed allocation has been implemented."
        ]
    }
    summary_bytes = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()

    for filename, records in (
        ("comparison_v1.csv", rows), ("per_layer_v1.csv", layer_rows)
    ):
        stream = io.StringIO(newline="")
        writer = csv.DictWriter(stream, fieldnames=list(records[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(records)
        outputs.append((filename, stream.getvalue().encode()))
    outputs.append(("summary_v1.json", summary_bytes))

    out.mkdir(parents=True, exist_ok=False)
    for filename, content in outputs:
        publish(out / filename, content)

    print("Reference masks reproduced exactly: PASS")
    print("Candidate | all-bit coverage % | storage overhead % | protected/byte | within budget")
    for row in rows:
        print(
            f"{row['candidate']} | {row['all_bit_coverage_pct']:.6f} | "
            f"{row['storage_overhead_pct']:.6f} | "
            f"{row['protected_positions_per_budget_byte']:.6f} | "
            f"{row['within_budget']}"
        )
    print("Coverage/cost nominee:", nominee)
    print("Saved:", out)
    print("Summary SHA256:", sha(summary_bytes))
    print("No COCO inference, AP selection, or changes to 05b.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Comparison has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/group_bit07a_compare_v1.py
Syntax: PASS
Comparison has not been executed.


In [8]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/group_bit07a_compare_v1.py"

spec = importlib.util.spec_from_file_location("group_bit07a_compare_v1", path)
comparison_module07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(comparison_module07a)

design_comparison07a = comparison_module07a.run(ROOT)

Reference masks reproduced exactly: PASS
Candidate | all-bit coverage % | storage overhead % | protected/byte | within budget
reference_g9_b456 | 1.454687 | 11.191998 | 1.039805 | True
g3_b4567 | 18.148379 | 33.428317 | 4.343235 | False
g9_b4567 | 2.658760 | 11.191854 | 1.900496 | True
g27_b4567 | 0.029220 | 3.760694 | 0.062159 | True
Coverage/cost nominee: g9_b4567
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/group_bit_design_comparison_v1
Summary SHA256: c7fb36fadf4083b0abe816fedbc3d1d4c73fb12649490eda5341400ccfa00423
No COCO inference, AP selection, or changes to 05b.


In [9]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/group_bit07a_nominee_check_v1.py"

source = r'''"""Functional checks for the frozen selected group-bit design."""
import hashlib
import importlib.util
import io
import json
from pathlib import Path

SUMMARY_SHA = "c7fb36fadf4083b0abe816fedbc3d1d4c73fb12649490eda5341400ccfa00423"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Payload checksum mismatch.")
    return envelope["payload"]

def correct(observed, packed, group_size, bits):
    """Inputs: observed int8 tensor and trusted masks; no clean target values."""
    import numpy as np
    if observed.dtype != np.int8 or packed.dtype != np.uint8:
        raise ValueError("Invalid dtypes.")
    result = observed.copy()
    o = result.shape[0]
    width = result.size // o
    end = (width // group_size) * group_size
    matrix = result.view(np.uint8).reshape(o, width)
    blocks = matrix[:, :end].reshape(-1, group_size)
    if packed.shape != (len(blocks),):
        raise ValueError("Mask length mismatch.")

    zero = np.zeros(len(packed), dtype=np.uint8)
    one = np.zeros(len(packed), dtype=np.uint8)
    for offset, bit in enumerate(bits):
        zero |= ((packed >> offset) & 1) << bit
        one |= ((packed >> (offset + len(bits))) & 1) << bit
    if np.any(zero & one):
        raise ValueError("Conflicting masks.")

    fixed = (blocks & np.bitwise_not(zero[:, None])) | one[:, None]
    matrix[:, :end] = fixed.reshape(o, end)
    return result

def run(repo):
    import numpy as np
    import torch
    repo = Path(repo)
    folder = repo / "results/notebook07a/group_bit_design_comparison_v1"
    destination = folder / "nominee_correctness_v1.json"
    if destination.exists():
        raise FileExistsError("Correctness report exists; inspect before rerunning.")

    summary = unpack(verified(folder / "summary_v1.json", SUMMARY_SHA))
    protocol = json.loads(verified(
        repo / "experiments/notebook07a/group_bit_design_comparison_protocol_v1.json",
        summary["protocol_sha256"]
    ))
    nominee = summary["nominee"]
    if nominee is None:
        raise RuntimeError("No eligible nominee.")
    candidate = next(c for c in protocol["candidates"] if c["name"] == nominee)
    descriptor_path = folder / f"{nominee}_descriptor.json"
    descriptor_bytes = descriptor_path.read_bytes()
    descriptor = json.loads(descriptor_bytes)
    if descriptor["candidate"] != candidate or descriptor["protocol_sha256"] != summary["protocol_sha256"]:
        raise RuntimeError("Descriptor provenance mismatch.")

    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    screen_path = repo / "src/screen05b_campaign_v1.py"
    verified(screen_path, plan["source_sha256"])
    spec = importlib.util.spec_from_file_location("screen07a_nominee", screen_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != summary["clean_model_state_sha256"]:
        raise RuntimeError("Clean model mismatch.")

    group_size, bits = candidate["group_size"], candidate["bits"]
    raw = verified(folder / descriptor["mask_archive"], descriptor["mask_archive_sha256"])
    total_weights = total_cases = recovered = 0

    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        if set(archive.files) != {r["array_key"] for r in descriptor["layers"]}:
            raise RuntimeError("Mask archive schema mismatch.")
        for item in descriptor["layers"]:
            q = prepared.quantized[item["layer"]][0].cpu().contiguous().numpy()
            if list(q.shape) != item["shape"]:
                raise RuntimeError("Layer shape mismatch.")
            packed = archive[item["array_key"]].copy()
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Layer mask checksum mismatch.")

            o = q.shape[0]
            width = q.size // o
            end = (width // group_size) * group_size
            blocks = q.view(np.uint8).reshape(o, width)[:, :end].reshape(-1, group_size)
            expected = np.zeros(len(blocks), dtype=np.uint8)
            protection = {}

            # Independently derive invariant bits from the clean groups.
            for offset, bit in enumerate(bits):
                values = (blocks >> bit) & 1
                zeros = np.all(values == 0, axis=1)
                ones = np.all(values == 1, axis=1)
                expected |= zeros.astype(np.uint8) << offset
                expected |= ones.astype(np.uint8) << (offset + len(bits))
                protection[bit] = zeros | ones
            if not np.array_equal(packed, expected):
                raise RuntimeError("Stored masks disagree with clean invariants.")
            if not np.array_equal(correct(q, packed, group_size, bits), q):
                raise RuntimeError("Clean weights changed.")

            for bit in range(8):
                faulty = (q.view(np.uint8) ^ np.uint8(1 << bit)).view(np.int8)
                final = correct(faulty, packed, group_size, bits)
                selected = np.zeros((o, width), dtype=bool)
                if bit in protection:
                    selected[:, :end] = np.repeat(
                        protection[bit], group_size
                    ).reshape(o, end)
                selected = selected.reshape(q.shape)
                if not np.array_equal(final[selected], q[selected]):
                    raise RuntimeError("Protected bit was not restored.")
                if not np.array_equal(final[~selected], faulty[~selected]):
                    raise RuntimeError("Unprotected bit or tail changed.")
                recovered += int(selected.sum())
                total_cases += q.size
            total_weights += q.size

    expected_row = next(row for row in summary["rows"] if row["candidate"] == nominee)
    if recovered != expected_row["protected_positions"]:
        raise RuntimeError("Recovery count differs from coverage report.")
    prepared._assert_state(prepared.clean_hash)

    payload = {
        "status": "NOMINEE_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION",
        "candidate": candidate,
        "comparison_summary_sha256": SUMMARY_SHA,
        "descriptor_sha256": sha(descriptor_bytes),
        "mask_archive_sha256": descriptor["mask_archive_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "clean_weights_unchanged": int(total_weights),
        "position_bit_cases_checked": int(total_cases),
        "protected_cases_exactly_recovered": int(recovered),
        "unprotected_cases_left_unchanged": int(total_cases - recovered),
        "limitations": [
            "Vectorized position-wise checks, not repeated network executions.",
            "Trusted reference masks, metadata, scales and checker.",
            "No AP benefit or deployment overhead established."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    print(json.dumps(payload, indent=2))
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("group_bit07a_nominee_check_v1", path)
nominee_check07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nominee_check07a)
nominee_correctness07a = nominee_check07a.run(ROOT)

{
  "status": "NOMINEE_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION",
  "candidate": {
    "name": "g9_b4567",
    "group_size": 9,
    "bits": [
      4,
      5,
      6,
      7
    ]
  },
  "comparison_summary_sha256": "c7fb36fadf4083b0abe816fedbc3d1d4c73fb12649490eda5341400ccfa00423",
  "descriptor_sha256": "22a669ad92fede091827051aafcfa7a82be95eb0ac96f37d35a486738ec45c47",
  "mask_archive_sha256": "895be126741b293b5c667993aba6821aa30eb0ce67bf5b03804b394d13eb04a9",
  "module_sha256": "723777c831121a64a38fe843c970300244495fcdd7e765bbd1b102cab86fd8eb",
  "clean_weights_unchanged": 11146080,
  "position_bit_cases_checked": 89168640,
  "protected_cases_exactly_recovered": 2370780,
  "unprotected_cases_left_unchanged": 86797860,
  "limitations": [
    "Vectorized position-wise checks, not repeated network executions.",
    "Trusted reference masks, metadata, scales and checker.",
    "No AP benefit or deployment overhead established."
  ]
}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Toleran

In [10]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/group_bit07a_archived_pilot_v1.py"

source = r'''"""Retrospective reuse of archived metrics for identical logical weight states."""
import gzip
import hashlib
import io
import json
from pathlib import Path

CHECK_SHA = "5c2bdc9e1f1b275b0627057b8cbcc45a670b0798c2cf454a8c4898f8f783de29"
DECISIONS_SHA = "89a9a3c3e2b4f8187089325a765b1a715b7059d40ed8446fe1f08e3881fab2c0"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Payload checksum mismatch.")
    return envelope["payload"]

def compressed_record(path):
    return unpack(gzip.decompress(path.read_bytes()))

def run(repo):
    import numpy as np
    repo = Path(repo)
    design = repo / "results/notebook07a/group_bit_design_comparison_v1"
    destination = design / "archived_16_scenario_pilot_v1.json"
    if destination.exists():
        raise FileExistsError("Report exists; inspect before rerunning.")

    checks = unpack(verified(design / "nominee_correctness_v1.json", CHECK_SHA))
    descriptor_bytes = verified(
        design / (checks["candidate"]["name"] + "_descriptor.json"),
        checks["descriptor_sha256"]
    )
    descriptor = json.loads(descriptor_bytes)
    if descriptor["candidate"] != checks["candidate"]:
        raise RuntimeError("Candidate mismatch.")
    raw_masks = verified(
        design / descriptor["mask_archive"], checks["mask_archive_sha256"]
    )
    layers = {row["layer"]: row for row in descriptor["layers"]}
    decisions = unpack(verified(
        repo / "results/notebook06b/end_to_end_decisions_v1.json", DECISIONS_SHA
    ))

    previous = repo / "results/notebook06b/end_to_end_pilot_v1"
    session = compressed_record(previous / "session.json.gz")
    fingerprint = sha(canonical(session))
    if session["decisions_sha256"] != DECISIONS_SHA:
        raise RuntimeError("Archived decisions mismatch.")

    protocol_path = repo / "experiments/notebook06b/end_to_end_pilot_protocol_v1.json"
    archived_protocol = json.loads(verified(protocol_path, session["protocol_sha256"]))
    comparison = unpack(verified(
        design / "summary_v1.json", checks["comparison_summary_sha256"]
    ))
    if comparison["clean_model_state_sha256"] != archived_protocol["clean_model_state_sha256"]:
        raise RuntimeError("Clean model reference mismatch.")

    metrics = {}
    evaluation_hashes = {}
    for index, state in enumerate(session["states"]):
        evaluation_path = previous / f"state_{index:03d}_evaluation.json.gz"
        evaluation = compressed_record(evaluation_path)
        prediction_path = previous / f"state_{index:03d}_predictions.json.gz"
        prediction_bytes = verified(prediction_path, evaluation["prediction_sha256"])
        prediction = unpack(gzip.decompress(prediction_bytes))
        if (
            evaluation["fingerprint"] != fingerprint
            or prediction["fingerprint"] != fingerprint
            or evaluation["index"] != index or prediction["index"] != index
            or evaluation["state"] != state
            or evaluation["state_sha256"] != prediction["state_sha256"]
            or prediction["data"]["image_ids"] != session["image_ids"]
            or evaluation["exact_clean_restoration"] is not True
        ):
            raise RuntimeError("Archived state provenance mismatch.")
        metrics[index] = evaluation["metrics"]
        evaluation_hashes[str(index)] = sha(evaluation_path.read_bytes())

    bits = checks["candidate"]["bits"]
    group_size = checks["candidate"]["group_size"]
    rows = []
    with np.load(io.BytesIO(raw_masks), allow_pickle=False) as archive:
        for item in decisions["rows"]:
            scenario = item["scenario"]
            sid = scenario["scenario_id"]
            layer = layers[scenario["layer"]]
            packed = archive[layer["array_key"]]
            if sha(packed.tobytes()) != layer["mask_array_sha256"]:
                raise RuntimeError("Layer mask mismatch.")

            o, i, kh, kw = layer["shape"]
            width = i * kh * kw
            flat = scenario["flat_index"]
            if not 0 <= flat < o * width:
                raise RuntimeError("Invalid target position.")
            channel, within = divmod(flat, width)
            if channel != scenario["channel"]:
                raise RuntimeError("Target channel mismatch.")
            complete = (width // group_size) * group_size
            zero = one = 0
            if within < complete:
                group_index = channel * (width // group_size) + within // group_size
                code = int(packed[group_index])
                for offset, bit in enumerate(bits):
                    zero |= ((code >> offset) & 1) << bit
                    one |= ((code >> (offset + len(bits))) & 1) << bit

            clean_unsigned = scenario["q_clean"] & 255
            faulty_unsigned = scenario["q_faulty"] & 255
            if (
                clean_unsigned & zero
                or (clean_unsigned & one) != one
                or zero & one
            ):
                raise RuntimeError("Masks disagree with clean target.")
            if clean_unsigned ^ faulty_unsigned != 1 << scenario["injected_bit"]:
                raise RuntimeError("Not the declared single-bit fault.")

            final_unsigned = (faulty_unsigned & (~zero & 255)) | one
            final = final_unsigned - 256 if final_unsigned >= 128 else final_unsigned
            clean_index = session["condition_states"][sid + "/shared_INT8_clean"]
            fault_index = session["condition_states"][sid + "/faulty_no_intervention"]

            if session["states"][clean_index] is not None:
                raise RuntimeError("Archived clean-state mapping mismatch.")
            if session["states"][fault_index] != [
                scenario["layer"], flat, scenario["q_faulty"]
            ]:
                raise RuntimeError("Archived fault-state mapping mismatch.")

            if final == scenario["q_clean"]:
                reuse_index = clean_index
            elif final == scenario["q_faulty"]:
                reuse_index = fault_index
            else:
                raise RuntimeError("New weight state requires new inference.")

            clean_ap = metrics[clean_index]["AP50_95"]
            fault_ap = metrics[fault_index]["AP50_95"]
            after_ap = metrics[reuse_index]["AP50_95"]
            rows.append({
                "scenario_id": sid,
                "layer": scenario["layer"],
                "bit": scenario["injected_bit"],
                "q_final": final,
                "exact_recovery": final == scenario["q_clean"],
                "reused_state_index": reuse_index,
                "clean_AP50_95": clean_ap,
                "faulty_AP50_95": fault_ap,
                "after_AP50_95": after_ap,
                "AP_gain_pp": 100 * (after_ap - fault_ap),
                "all_12_metrics_after": metrics[reuse_index]
            })

    gains = [row["AP_gain_pp"] for row in rows]
    payload = {
        "status": "RETROSPECTIVE_ARCHIVED_STATE_REUSE_NOT_NEW_INFERENCE",
        "candidate": checks["candidate"],
        "nominee_correctness_sha256": CHECK_SHA,
        "descriptor_sha256": checks["descriptor_sha256"],
        "archived_session_fingerprint": fingerprint,
        "evaluation_sha256_by_state": evaluation_hashes,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "scenario_count": len(rows),
        "exact_recoveries": sum(row["exact_recovery"] for row in rows),
        "mean_signed_AP_gain_pp": sum(gains) / len(gains),
        "positive_gain_scenarios": sum(value > 0 for value in gains),
        "negative_gain_scenarios": sum(value < 0 for value in gains),
        "rows": rows,
        "limitations": [
            "Previously inspected 16 development scenarios; retrospective only.",
            "Metrics reused for identical logical clean or single-weight fault states.",
            "No fresh CPU or GPU model inference; no new COCOeval run.",
            "Current image files and reconstructed model states were not rechecked here.",
            "State equivalence relies on validated clean-mask invariance and single-weight fault scope.",
            "No independent AP efficacy or novelty confirmation."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    print("Scenario | exact recovery | AP gain, percentage points")
    for row in rows:
        print(
            f"{row['scenario_id']} | {row['exact_recovery']} | "
            f"{row['AP_gain_pp']:+.8f}"
        )
    print("Exact recoveries:", payload["exact_recoveries"], "/", len(rows))
    print("Mean signed AP gain, pp:", payload["mean_signed_AP_gain_pp"])
    print("Saved:", destination)
    print("SHA256:", sha(content))
    print("No inference, parameter changes, or 05b campaign modifications.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("group_bit07a_archived_pilot_v1", path)
archived_pilot07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(archived_pilot07a)
archived_pilot_result07a = archived_pilot07a.run(ROOT)

Scenario | exact recovery | AP gain, percentage points
S06B-E2E-001 | False | +0.00000000
S06B-E2E-002 | False | +0.00000000
S06B-E2E-003 | False | +0.00000000
S06B-E2E-004 | False | +0.00000000
S06B-E2E-005 | False | +0.00000000
S06B-E2E-006 | False | +0.00000000
S06B-E2E-007 | False | +0.00000000
S06B-E2E-008 | False | +0.00000000
S06B-E2E-009 | False | +0.00000000
S06B-E2E-010 | False | +0.00000000
S06B-E2E-011 | False | +0.00000000
S06B-E2E-012 | False | +0.00000000
S06B-E2E-013 | False | +0.00000000
S06B-E2E-014 | False | +0.00000000
S06B-E2E-015 | False | +0.00000000
S06B-E2E-016 | False | +0.00000000
Exact recoveries: 0 / 16
Mean signed AP gain, pp: 0.0
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/group_bit_design_comparison_v1/archived_16_scenario_pilot_v1.json
SHA256: fefcc795bf93991f134248023d59a3c0bee22f1140cf82741d439bf879c537cd
No inference, parameter changes, or 05b campaign modifications.


In [11]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

for args in (
    ["rev-parse", "HEAD"],
    ["--no-optional-locks", "status", "--short", "--untracked-files=normal"],
    ["diff", "--stat"],
):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=120,
    )
    print("\nCHECK:", " ".join(args))
    print("Return code:", result.returncode)
    print(result.stdout.strip() or "(no output)")
    if result.stderr.strip():
        print(result.stderr.strip())

print("\nNo staging, commit, push, or file changes.")


CHECK: rev-parse HEAD
Return code: 0
1155a56eae3accdfe3b4b820cdc3d539e9da958c

CHECK: --no-optional-locks status --short --untracked-files=normal
Return code: 0
?? docs/research_reports/notebook05b_cpu_replay_v1/
?? docs/research_reports/notebook05b_screening_v1/progress_01750.json
?? docs/research_reports/notebook05b_screening_v1/progress_01800.json
?? docs/research_reports/notebook05b_screening_v1/progress_01850.json
?? docs/research_reports/notebook05b_screening_v1/progress_01900.json
?? docs/research_reports/notebook05b_screening_v1/progress_01950.json
?? docs/research_reports/notebook05b_screening_v1/progress_02000.json
?? docs/research_reports/notebook05b_screening_v1/progress_02050.json
?? docs/research_reports/notebook05b_screening_v1/progress_02100.json
?? docs/research_reports/notebook05b_screening_v1/progress_02150.json
?? experiments/notebook07a/group_bit_design_comparison_protocol_v1.json
?? src/group_bit07a_archived_pilot_v1.py
?? src/group_bit07a_compare_v1.py
?? src/gr